# Track A 폴더9 — STEP 17~24: 강사 피드백 대응 검증 노트북
작성: 2026-07-22 | 전제: 기존 노트북의 STEP 11(코호트 ckpt)·STEP 12(배치 스캔)·STEP 15-4(최종목록)가 Drive `ckpt/`에 저장돼 있음

**원본 CSV를 다시 읽지 않는다.** 모든 입력은 체크포인트(parquet)에서 복원한다.

| STEP | 내용 | 대응 피드백 |
|---|---|---|
| 17 | 환경 복원 (ckpt 로드, 와이드 프레임, IV 함수) | — |
| 18 | 타겟 창 24개월 확장 (양성 증대) | 문제 1·2·3 |
| 19-1 | 신뢰수준 → 품질 플래그 재배치 + 결합 파생 심사 | 문제 2 |
| 19-2 | 자산 7종 → 대표 2~3개 축소 | 문제 2 |
| 19-3 | Δ생활이력(패널 차분) 신규 파생 심사 | 문제 2 |
| 19-4 | 차량 보유 플래그·미검토 컬럼 확인 | 문제 2 |
| 20 | 반복 CV + OOF 부트스트랩 95% CI | 문제 1·2(신뢰구간) |
| 21 | 불균형 처리 4방법 비교표 | 문제 3 |
| 22 | 2022 급증 분해 (상품·채널·인구·SCORE) | 문제 4 |
| 23 | Leave-One-Year-Out + 2022 다운웨이트 민감도 | 문제 4 |
| 24 | 구간화 CSV + WOE 유틸 (모델링 담당 전달물) | 문제 2 |

## STEP 17. 환경 복원
런타임이 끊겨도 이 셀부터 다시 실행하면 된다. 필요한 ckpt: `cohort_entrant_ids_allyears`, `batch_*.parquet`(7개), `step15_4_최종대안변수목록`.

In [1]:
# ============ STEP 17. 환경 복원 — 이 노트북은 단독 실행 가능 (원본 CSV 재로드 불필요) ============
import os, gc, json, glob, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)
RANDOM_STATE = 42

try:                                     # Colab이면 Drive 마운트
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/09.개인_CB정보'
except ImportError:                      # 로컬 테스트 폴백
    DATA_DIR = os.environ.get('DATA_DIR', '.')

CKPT_DIR = os.path.join(DATA_DIR, 'ckpt')

def save_ckpt(frame, name):
    path = os.path.join(CKPT_DIR, f'{name}.parquet'); frame.to_parquet(path, index=False)
    print(f'[ckpt 저장] {name}: {len(frame):,}행 → {path}')

def load_ckpt(name):
    frame = pd.read_parquet(os.path.join(CKPT_DIR, f'{name}.parquet'))
    print(f'[ckpt 로드] {name}: {len(frame):,}행'); return frame

def ckpt_exists(name): return os.path.exists(os.path.join(CKPT_DIR, f'{name}.parquet'))

# --- 기존 산출물 로드 ---
ent_all   = load_ckpt('cohort_entrant_ids_allyears')       # 진입자 ID x 4개년 (STEP 11 산출)
final     = load_ckpt('step15_4_최종대안변수목록')          # STEP 15-4 산출
ALT_FINAL = final['col'].tolist()
print('STEP 15 최종 대안변수:', ALT_FINAL)

# --- 배치 parquet 병합 → 전 연도 와이드 프레임 (13R과 동일 로직) ---
wide_all = ent_all.copy(); wide_all['ID'] = wide_all['ID'].astype(str)
for path in sorted(glob.glob(os.path.join(CKPT_DIR, 'batch_*.parquet'))):
    b = pd.read_parquet(path); b['ID'] = b['ID'].astype(str)
    add = ['ID','YEAR'] + [c for c in b.columns if c not in wide_all.columns]
    wide_all = wide_all.merge(b[add], on=['ID','YEAR'], how='left')
print(f'전 연도 프레임: {wide_all.shape[0]:,}행 × {wide_all.shape[1]}열')

# --- NONBANK_RATIO (13R과 동일) ---
NB_NUM = ['L10210800','L10210B00','L90210300','L90210200','L10210M00']
_num = wide_all[NB_NUM].fillna(0).clip(lower=0).sum(axis=1)
_den = wide_all['L10210000'].fillna(0)
wide_all['NONBANK_RATIO'] = np.where(_den > 0, (_num/_den).clip(0, 1), -1)   # -1 = 대출없음 범주

# --- 공용 IV 함수 (기존 노트북과 동일 로직) ---
def iv_cat(x, y, min_n=30):
    x = pd.Series(x).astype(str).fillna('NULL')
    vc = x.value_counts(); x = x.where(x.map(vc) >= min_n, '기타')
    g = pd.DataFrame({'x': x.values, 'y': np.asarray(y)}).groupby('x')['y'].agg(n='size', bad='sum')
    g['good'] = g['n'] - g['bad']; tb, tg = g['bad'].sum(), g['good'].sum()
    if tb == 0 or tg == 0: return 0.0, g.assign(bad_rate_pct=0)
    br, gr = (g['bad']+.5)/(tb+.5), (g['good']+.5)/(tg+.5)
    g['bad_rate_pct'] = (g['bad']/g['n']*100).round(3); g['WoE'] = np.log(gr/br).round(3)
    return float(((gr-br)*np.log(gr/br)).sum()), g

def iv_num(x, y, n_bins=5):
    x = pd.Series(x).astype(float).fillna(0)
    grp = pd.Series('zero', index=x.index); nz = x != 0
    if nz.sum() >= n_bins*20:
        try: grp[nz] = pd.qcut(x[nz], q=n_bins, duplicates='drop').astype(str)
        except ValueError: grp[nz] = 'nonzero'
    elif nz.any(): grp[nz] = 'nonzero'
    iv, _ = iv_cat(grp, y, min_n=1)
    return iv


Mounted at /content/drive
[ckpt 로드] cohort_entrant_ids_allyears: 273,468행
[ckpt 로드] step15_4_최종대안변수목록: 14행
STEP 15 최종 대안변수: ['AL012G019', 'U81301010', 'U81305010', 'AS120G001', 'U81306010', 'U81302010', 'U81201010', 'U81202010', 'U81102010', 'AL012G005', 'U81304010', 'U81303010', 'U81205010', 'AL012G011']
전 연도 프레임: 273,468행 × 175열


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## STEP 18. [레버 2] 타겟 창 24개월 확장
모집단(진입자)은 그대로 두고 성과창만 넓혀 양성을 늘린다 — "무이력 대안평가"라는 프로젝트 정체성을 지키는 표본 확장.
2022 진입자는 검열(12M) — 12M/24M 병행 보고가 원칙.

In [4]:
# ============ STEP 18. [레버 2] 타겟 창 확장 — PERF2(t) ∪ PERF2(t+1) = "진입 후 24개월 내 연체" ============
# 근거: PERF2는 전방 12개월 라벨(STEP 5 검증). 패널에 진입자의 t+1년 행이 있으므로
#       t와 t+1의 PERF2를 OR로 묶으면 성과창이 24개월로 늘어 양성이 증가한다.
# 제약: 2022 진입자는 2023 스냅샷이 없어 24M 창이 12M으로 검열(censoring)됨 → 반드시 명시.

ent_t = wide_all[wide_all['IS_ENTRANT']].copy().reset_index(drop=True)

nxt = wide_all[['ID','YEAR','PERF2']].copy()
nxt['YEAR'] = nxt['YEAR'] - 1                      # t+1년 행을 t년 키로 당겨오기
nxt = nxt.rename(columns={'PERF2': 'PERF2_next'})
ent_t = ent_t.merge(nxt, on=['ID','YEAR'], how='left')

ent_t['TARGET_12M'] = ent_t['PERF2'].astype(int)
ent_t['HAS_NEXT']   = ent_t['PERF2_next'].notna()
ent_t['TARGET_24M'] = ((ent_t['PERF2'] == 1) | (ent_t['PERF2_next'] == 1)).astype(int)

rep = ent_t.groupby('YEAR').agg(n=('ID', 'size'), pos_12M=('TARGET_12M', 'sum'),
                                pos_24M=('TARGET_24M', 'sum'), 관측창보유율=('HAS_NEXT', 'mean')).round(3)
rep['성과창'] = np.where(rep['관측창보유율'] > 0, '24개월', '12개월(검열)')
display(rep)
g12, g24 = int(ent_t['TARGET_12M'].sum()), int(ent_t['TARGET_24M'].sum())
print(f'양성: 12M {g12}건 → 24M {g24}건 (+{g24-g12}건, {g24/max(g12,1):.2f}배)')
print('보고 문구: "성과창 24개월 확장 시 2022 진입자는 관측창 12개월로 검열 — 연도별 관측창 차이를 명시하고'
      ' 12M/24M 두 타겟을 병행 보고(민감도 분석)"')


,n,pos_12M,pos_24M,관측창보유율,성과창
YEAR,,,,,
2020,8150,43,48,1.0,24개월
2021,8610,7,50,1.0,24개월
2022,51917,46,46,0.0,12개월(검열)


양성: 12M 96건 → 24M 144건 (+48건, 1.50배)
보고 문구: "성과창 24개월 확장 시 2022 진입자는 관측창 12개월로 검열 — 연도별 관측창 차이를 명시하고 12M/24M 두 타겟을 병행 보고(민감도 분석)"


## STEP 19. 변수 정비 (신뢰수준·자산·Δ파생·차량)

In [5]:
# ============ STEP 19-1. 신뢰수준(A/B/C) 재배치 — 독립변수 제외 → 품질 플래그 + 파생 결합 검증 ============
# 논리: 신뢰수준은 위험 자체가 아니라 "짝이 되는 가격 추정치의 품질" 메타정보.
#       ① 독립변수에서 제외 ② '신뢰정보 없음' 플래그로 접기 ③ 가격×신뢰 결합 파생은
#          기존 규칙("원변수보다 IV 높을 때만 채택") 그대로 심사.

y12 = ent_t['TARGET_12M'].values

CONF_PAIRS = {                    # 신뢰수준 → 짝 가격/자산 변수
    'U81303010': 'U81301010',    # 거주지매매가신뢰수준 → 거주지매매가
    'U81304010': 'U81302010',    # 거주지전세가신뢰수준 → 거주지전세가
    'U81205010': 'U81201010',    # 자산평가신뢰수준 → 총자산평가금액
}
rows = []
for conf, price in CONF_PAIRS.items():
    if conf not in ent_t.columns or price not in ent_t.columns:
        print(f'{conf}/{price} 없음 — 건너뜀'); continue
    c = ent_t[conf].astype(str).str.strip().str.upper()
    flag = (~c.isin(['A', 'B', 'C'])).astype(int)              # N·결측·기타 = 신뢰정보 없음
    ent_t[f'{price}_CONF_MISSING'] = flag

    iv_conf, _ = iv_cat(c, y12, min_n=30)
    iv_flag, _ = iv_cat(flag, y12, min_n=1)
    iv_price   = iv_num(ent_t[price], y12)

    # 파생 결합: 가격 5분위(0-분리) × 신뢰수준
    p = pd.to_numeric(ent_t[price], errors='coerce').fillna(0)
    pb = pd.Series('zero', index=p.index, dtype=object); nz = p != 0
    if nz.sum() >= 100:
        try: pb[nz] = pd.qcut(p[nz], q=5, duplicates='drop').astype(str)
        except ValueError: pb[nz] = 'nonzero'
    iv_combo, _ = iv_cat(pb + '|' + c, y12, min_n=30)

    verdict = '파생 후보(모델 기여도로 최종 확인)' if iv_combo > 1.2 * max(iv_price, iv_conf) \
              else '가격 원변수만 유지, 파생 폐기'
    rows.append({'신뢰수준': conf, '가격변수': price,
                 'IV(신뢰 단독)': round(iv_conf, 4), 'IV(가격)': round(iv_price, 4),
                 'IV(품질플래그)': round(iv_flag, 4), 'IV(가격×신뢰 결합)': round(iv_combo, 4),
                 '판정': verdict})
res_conf = pd.DataFrame(rows); display(res_conf)
save_ckpt(res_conf, 'step19_1_신뢰수준재배치')
print('★ 주의: 결합 파생은 범주 수(5분위×4등급=최대 20개)가 많아 IV가 "기계적으로" 부푼다 — NONBANK_RATIO 0.886과 같은 계열의 착시.')
print('  그래서 판정 임계를 1.2배로 올렸고, "후보" 판정이 나와도 STEP 20의 fold별 모델 기여도로 최종 확인해야 채택.')
print('→ 대안변수 문서 3·7절 수정 근거: 신뢰수준 3종은 독립변수 목록에서 빼고 *_CONF_MISSING 플래그로 대체.')


,신뢰수준,가격변수,IV(신뢰 단독),IV(가격),IV(품질플래그),IV(가격×신뢰 결합),판정
0,U81303010,U81301010,0.0489,0.3888,0.0352,0.4613,"가격 원변수만 유지, 파생 폐기"
1,U81304010,U81302010,0.0540,0.2456,0.0053,0.3491,파생 후보(모델 기여도로 최종 확인)
2,U81205010,U81201010,0.0372,0.2303,0.0044,0.3697,파생 후보(모델 기여도로 최종 확인)


[ckpt 저장] step19_1_신뢰수준재배치: 3행 → /content/drive/MyDrive/09.개인_CB정보/ckpt/step19_1_신뢰수준재배치.parquet
★ 주의: 결합 파생은 범주 수(5분위×4등급=최대 20개)가 많아 IV가 "기계적으로" 부푼다 — NONBANK_RATIO 0.886과 같은 계열의 착시.
  그래서 판정 임계를 1.2배로 올렸고, "후보" 판정이 나와도 STEP 20의 fold별 모델 기여도로 최종 확인해야 채택.
→ 대안변수 문서 3·7절 수정 근거: 신뢰수준 3종은 독립변수 목록에서 빼고 *_CONF_MISSING 플래그로 대체.


In [6]:
# ============ STEP 19-2. 자산·거주 7종 → 대표 2~3개 축소 (양성 96건 체제의 차원 절약) ============
# 13R 중요도 상위가 전부 연속 U8 자산변수였던 것 = 소표본에서 연속변수 노이즈 분기(과적합 신호).
# 상관 0.8+ 그리디(IV 내림차순)로 대표만 남긴다. 다중공선성 '완전 박멸'이 목적이 아니라 차원 축소가 목적.

ASSET7 = [c for c in ['U81301010','U81305010','U81306010','U81302010','U81201010','U81202010','U81102010']
          if c in ent_t.columns]
Xa   = ent_t[ASSET7].apply(pd.to_numeric, errors='coerce').fillna(0)
ivs  = {c: iv_num(Xa[c], y12) for c in ASSET7}
corr = Xa.corr().abs()

keep, dropped = [], set()
for c in sorted(ASSET7, key=lambda c: -ivs[c]):
    if c in dropped: continue
    keep.append(c)
    for other in ASSET7:
        if other != c and other not in dropped and corr.loc[c, other] >= 0.8:
            dropped.add(other)
ASSET_KEEP = keep[:3]
print('IV 내림차순:', {k: round(v, 3) for k, v in sorted(ivs.items(), key=lambda x: -x[1])})
print('상관 0.8+ 제거:', sorted(dropped))
print('대표 채택(최대 3):', ASSET_KEEP)


IV 내림차순: {'U81301010': 0.389, 'U81305010': 0.284, 'U81306010': 0.248, 'U81302010': 0.246, 'U81201010': 0.23, 'U81202010': 0.226, 'U81102010': 0.18}
상관 0.8+ 제거: []
대표 채택(최대 3): ['U81301010', 'U81305010', 'U81306010']


In [7]:
# ============ STEP 19-3. [신규 파생] Δ생활이력 — 패널 연차 차분 = "최근 1년간 변경 건수" ============
# AL012G는 '3년내 누적' 구간화 값. 패널이므로 연차 차분(t − t-1)이 "최근 1년 변화"의 근사가 된다.
# (구간화 값의 차분이라 정확한 건수 차는 아님 — 근사 지표임을 문서에 명시)
# A-1용 Δ = t − (t-1) : 전 진입자 가용.
# A-2용 Δ = (t-1) − (t-2) : 2020 진입자는 2018 스냅샷이 없어 결측 → 가용률 확인 후 채택 결정.

AL3 = ['AL012G005', 'AL012G011', 'AL012G019']

for lag, suf in [(1, '_tm1'), (2, '_tm2')]:
    if f'{AL3[0]}{suf}' in ent_t.columns: continue          # 재실행 가드
    prv = wide_all[['ID', 'YEAR'] + AL3].copy(); prv['YEAR'] = prv['YEAR'] + lag
    ent_t = ent_t.merge(prv.rename(columns={c: f'{c}{suf}' for c in AL3}), on=['ID','YEAR'], how='left')

rows = []
for c in AL3:
    d1 = (pd.to_numeric(ent_t[c], errors='coerce') - pd.to_numeric(ent_t[f'{c}_tm1'], errors='coerce')).clip(lower=0)
    d2 = (pd.to_numeric(ent_t[f'{c}_tm1'], errors='coerce') - pd.to_numeric(ent_t[f'{c}_tm2'], errors='coerce')).clip(lower=0)
    ent_t[f'DELTA_{c}']      = d1        # A-1용
    ent_t[f'DELTA_{c}_prev'] = d2        # A-2용
    iv_orig = iv_cat(ent_t[c], y12)[0]
    iv_d1   = iv_cat(d1.fillna(-1), y12)[0]
    m2 = d2.notna()
    iv_d2   = iv_cat(d2[m2], y12[m2.values])[0] if m2.sum() > 1000 else np.nan
    rows.append({'변수': c, 'IV(누적3년, t)': round(iv_orig, 4), 'IV(Δ, A-1용)': round(iv_d1, 4),
                 'IV(Δ, A-2용)': (round(iv_d2, 4) if iv_d2 == iv_d2 else 'n/a'),
                 'A-2 Δ 가용률': round(float(m2.mean()), 3)})
res_delta = pd.DataFrame(rows); display(res_delta)
save_ckpt(res_delta, 'step19_3_델타생활이력')
print('판정 규칙 동일: Δ가 누적치 IV를 이기면 채택, 못 이기면 폐기(파생 심사 규칙 4절과 동일).')


,변수,"IV(누적3년, t)","IV(Δ, A-1용)","IV(Δ, A-2용)",A-2 Δ 가용률
0,AL012G005,0.3643,0.0607,0.4189,0.881
1,AL012G011,0.1259,0.1177,0.0310,0.881
2,AL012G019,0.5405,0.0875,0.2489,0.881


[ckpt 저장] step19_3_델타생활이력: 3행 → /content/drive/MyDrive/09.개인_CB정보/ckpt/step19_3_델타생활이력.parquet
판정 규칙 동일: Δ가 누적치 IV를 이기면 채택, 못 이기면 폐기(파생 심사 규칙 4절과 동일).


In [8]:
# ============ STEP 19-4. 차량 보유 플래그 + 미검토 컬럼(U10000002 외국인여부, U81103010) 확인 ============
def has_info(s):
    """정보없음(-9, NULL, 결측) 여부 → 1=정보 있음"""
    st = s.astype(str).str.strip().str.upper()
    return (~(s.isna() | st.isin(['-9', '-9.0', 'NULL', 'NAN', 'NONE', '']))).astype(int)

# ① 차량: 개별 5종은 커버리지(96%+ 정보없음)로 기각됐지만, '보유 여부' 이진 플래그는 커버리지 문제를 우회
if 'AL0C00001' in ent_t.columns:
    ent_t['CAR_FLAG'] = has_info(ent_t['AL0C00001'])
    iv, tbl = iv_cat(ent_t['CAR_FLAG'], y12, min_n=1)
    print(f"[CAR_FLAG] 보유율 {ent_t['CAR_FLAG'].mean():.3%} | IV {iv:.4f}")
    display(tbl)
else:
    print('AL0C00001이 배치 parquet에 없음 — STEP 12 스캔 컬럼 확인 필요')

# ② 확정/기각 어느 목록에도 없던 컬럼 — 배치 스캔 결과에서 판정만 확인
for c in ['U10000002', 'U81103010']:
    if c in ent_t.columns:
        na_pct = (1 - has_info(ent_t[c]).mean()) * 100
        iv, tbl = iv_cat(ent_t[c], y12, min_n=30)
        print(f'[{c}] 정보없음 {na_pct:.1f}% | IV(범주형) {iv:.4f}')
        display(tbl.head(8))
    else:
        print(f'{c}: 배치 parquet에 없음 — iv_scan_remain128_entrant 체크포인트에서 재확인')


[CAR_FLAG] 보유율 3.110% | IV 0.1505


,n,bad,good,bad_rate_pct,WoE
x,,,,,
0,66541,84,66457,0.126,0.101
1,2136,12,2124,0.562,-1.431


[U10000002] 정보없음 0.0% | IV(범주형) 0.0000


,n,bad,good,bad_rate_pct,WoE
x,,,,,
0.0,68677,96,68581,0.14,0.0


[U81103010] 정보없음 66.9% | IV(범주형) 0.0166


,n,bad,good,bad_rate_pct,WoE
x,,,,,
A,17653,30,17623,0.170,-0.207
B,4257,5,4252,0.117,0.084
C,799,1,798,0.125,-0.289
None,45968,60,45908,0.131,0.066


## STEP 20. [레버 4] 반복 CV + pooled OOF 부트스트랩 CI
fold 표준편차(±) 표기를 폐기하고 정식 95% CI로 교체. 핵심 판정은 **CI 하한이 0.5를 넘는가**.

In [9]:
# ============ STEP 20. [레버 4] 반복 CV + pooled OOF 부트스트랩 CI — 13R의 성능 보고 대체 ============
# fold 5개의 평균±표준편차는 신뢰구간이 아니다(표본 5개 + fold 간 비독립).
# 교체: ① 반복 StratifiedGroupKFold(5-fold × 10 seed)로 추정 분포 확인
#       ② out-of-fold 예측을 모아 ID(사람) 단위 클러스터 부트스트랩 1,000회 → 정식 95% CI

import lightgbm as lgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score

# --- t-1(_prev) 대안변수 병합 (13R과 동일, 재실행 가드) ---
PREV_COLS = [c for c in ALT_FINAL if c in wide_all.columns]
if f'{PREV_COLS[0]}_prev' not in ent_t.columns:
    prv = wide_all[['ID', 'YEAR'] + PREV_COLS].copy(); prv['YEAR'] = prv['YEAR'] + 1
    ent_t = ent_t.merge(prv.rename(columns={c: f'{c}_prev' for c in PREV_COLS}), on=['ID','YEAR'], how='left')

def encode(frame, cols):
    X = pd.DataFrame(index=frame.index)
    for c in cols:
        s = frame[c]
        if pd.api.types.is_numeric_dtype(s):
            X[c] = pd.to_numeric(s, errors='coerce').fillna(0)
        else:
            X = pd.concat([X, pd.get_dummies(s.astype(str), prefix=c)], axis=1)
    return X

# --- 피처셋 v2: 신뢰수준 제외·자산 대표 축소·플래그·Δ 반영 ---
CONF3   = [c for c in ['U81303010','U81304010','U81205010']]
ASSET7  = ['U81301010','U81305010','U81306010','U81302010','U81201010','U81202010','U81102010']
BASE_V2 = [c for c in ALT_FINAL if c not in CONF3 and (c not in ASSET7 or c in ASSET_KEEP)]
FLAGS   = [c for c in ent_t.columns if c.endswith('_CONF_MISSING')] + (['CAR_FLAG'] if 'CAR_FLAG' in ent_t.columns else [])
AL3     = ['AL012G005', 'AL012G011', 'AL012G019']

X1 = encode(ent_t, BASE_V2 + FLAGS + [f'DELTA_{c}' for c in AL3])
X1['NONBANK_RATIO'] = ent_t['NONBANK_RATIO']                       # A-1 전용(신청정보)
X1 = pd.concat([X1, pd.get_dummies(ent_t['GENDER'], prefix='GENDER'),
                    pd.get_dummies(ent_t['AGE_BAND'], prefix='AGE')], axis=1)

BASE_V2_PREV = [f'{c}_prev' for c in BASE_V2 if f'{c}_prev' in ent_t.columns]
X2 = encode(ent_t, BASE_V2_PREV + [f'DELTA_{c}_prev' for c in AL3])
X2 = pd.concat([X2, pd.get_dummies(ent_t['GENDER'], prefix='GENDER'),
                    pd.get_dummies(ent_t['AGE_BAND'], prefix='AGE')], axis=1)
print(f'피처 차원: A-1 {X1.shape[1]} | A-2 {X2.shape[1]} (EPV 참고: 양성 96건 → 유효 피처 5~10개 권장)')

groups = ent_t['ID'].values

LGB_SMALL = dict(n_estimators=300, learning_rate=0.05, max_depth=3, num_leaves=7,
                 min_child_samples=200, colsample_bytree=0.7, reg_lambda=5.0,
                 random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)   # 소표본용 강한 정규화 (기존 num_leaves=15 대비 축소)

def repeated_cv_oof(X, y, groups, n_repeats=10, n_splits=5, params=None, sample_weight=None):
    """반복 StratifiedGroupKFold → (반복별 OOF AUROC 분포, 반복 평균 OOF 예측)"""
    params = params or LGB_SMALL
    oof_sum = np.zeros(len(y)); rep_auc = []
    for r in range(n_repeats):
        sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=100 + r)
        oof = np.full(len(y), np.nan)
        for tr, va in sgkf.split(X, y, groups):
            spw = (y[tr] == 0).sum() / max((y[tr] == 1).sum(), 1)
            m = lgb.LGBMClassifier(scale_pos_weight=spw, **params)
            m.fit(X.iloc[tr], y[tr], sample_weight=None if sample_weight is None else sample_weight[tr])
            oof[va] = m.predict_proba(X.iloc[va])[:, 1]
        rep_auc.append(roc_auc_score(y, oof)); oof_sum += oof
    return np.array(rep_auc), oof_sum / n_repeats

def cluster_bootstrap_ci(y, p, groups, n_boot=1000, seed=42):
    """ID 단위 복원추출 부트스트랩 → (AUROC mean, lo, hi), (AUPRC mean, lo, hi)"""
    rng = np.random.default_rng(seed)
    idx_by_id = pd.Series(np.arange(len(y))).groupby(pd.Series(groups)).apply(lambda s: s.to_numpy())
    n_ids = len(idx_by_id); aucs, aps = [], []
    for _ in range(n_boot):
        pick = rng.integers(0, n_ids, size=n_ids)
        bi = np.concatenate(idx_by_id.iloc[pick].to_numpy())
        yb, pb = y[bi], p[bi]
        if yb.sum() in (0, len(yb)): continue
        aucs.append(roc_auc_score(yb, pb)); aps.append(average_precision_score(yb, pb))
    (lo, hi), (alo, ahi) = np.percentile(aucs, [2.5, 97.5]), np.percentile(aps, [2.5, 97.5])
    return (float(np.mean(aucs)), float(lo), float(hi)), (float(np.mean(aps)), float(alo), float(ahi))

# results = []
# for label, X in [('A-1(진입시점)', X1), ('A-2(진입전, 순수대안)', X2)]:
#     for tname in ['TARGET_12M', 'TARGET_24M']:
#         y = ent_t[tname].astype(int).values
#         rep_auc, oof = repeated_cv_oof(X, y, groups)
#         (am, alo, ahi), (pm, plo, phi) = cluster_bootstrap_ci(y, oof, groups)
#         results.append({'모델': label, '타겟': tname, '양성': int(y.sum()),
#                         'AUROC(OOF)': round(roc_auc_score(y, oof), 4),
#                         'AUROC 95% CI': f'[{alo:.3f}, {ahi:.3f}]',
#                         '반복CV 분포': f'{rep_auc.mean():.3f} ({rep_auc.min():.3f}~{rep_auc.max():.3f})',
#                         'AUPRC(OOF)': round(average_precision_score(y, oof), 4),
#                         'AUPRC 95% CI': f'[{plo:.4f}, {phi:.4f}]',
#                         '기준선(양성률)': round(float(y.mean()), 4)})
# res20 = pd.DataFrame(results); display(res20)
# save_ckpt(res20, 'step20_반복CV_부트스트랩CI')
# print('보고 문구 예: "AUROC 0.63 (95% CI 0.55–0.71; ID-클러스터 부트스트랩 1,000회, OOF 기준)"')
# print('핵심 판정: CI 하한이 0.5를 넘는가 — 넘으면 "무이력 정보에 변별력 존재"가 통계적으로 성립.')


피처 차원: A-1 26 | A-2 21 (EPV 참고: 양성 96건 → 유효 피처 5~10개 권장)


## STEP 21. 불균형 처리 비교 실험
강사가 언급한 옵션(모델 가중 / 샘플링 / 차원축소)을 전부 fold 내부에서 실측 비교 — 원칙을 결과로 방어.

In [10]:
# ============ STEP 21. 불균형 처리 비교 실험 — 강사 피드백(모델 vs 샘플링 vs 차원축소) 대응표 ============
# 원칙(SMOTE 금지)은 '선언'이 아니라 '실측'으로 방어한다: 전부 fold 내부에서만 적용해 한 표로 비교.
from sklearn.decomposition import PCA
try:
    from imblearn.over_sampling import SMOTE
    HAS_SMOTE = True
except ImportError:
    HAS_SMOTE = False; print('imblearn 미설치 → !pip install imbalanced-learn 후 재실행')

y_exp = ent_t['TARGET_24M'].astype(int).values      # 양성이 더 많은 24M 기준 (12M로 바꿔 재실행 가능)
X_exp = X2                                           # A-2(순수 대안평가) 기준

def cv_oof_custom(X, y, groups, fit_predict, n_splits=5, seed=42):
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.full(len(y), np.nan)
    for tr, va in sgkf.split(X, y, groups):
        oof[va] = fit_predict(X.iloc[tr], y[tr], X.iloc[va])
    return oof

def fp_spw(Xtr, ytr, Xva):
    spw = (ytr == 0).sum() / max(ytr.sum(), 1)
    m = lgb.LGBMClassifier(scale_pos_weight=spw, **LGB_SMALL); m.fit(Xtr, ytr)
    return m.predict_proba(Xva)[:, 1]

def fp_downsample_ens(Xtr, ytr, Xva, ratio=20, n_models=5):
    pos, neg = np.where(ytr == 1)[0], np.where(ytr == 0)[0]; preds = []
    for s in range(n_models):
        rng = np.random.default_rng(s)
        sel = np.concatenate([pos, rng.choice(neg, size=min(len(neg), len(pos) * ratio), replace=False)])
        m = lgb.LGBMClassifier(**LGB_SMALL); m.fit(Xtr.iloc[sel], ytr[sel])
        preds.append(m.predict_proba(Xva)[:, 1])
    return np.mean(preds, axis=0)

def fp_smote(Xtr, ytr, Xva):
    k = int(min(5, max(ytr.sum() - 1, 1)))
    Xs, ys = SMOTE(random_state=42, k_neighbors=k).fit_resample(Xtr.fillna(0), ytr)
    m = lgb.LGBMClassifier(**LGB_SMALL); m.fit(Xs, ys)
    return m.predict_proba(Xva)[:, 1]

def fp_spw_pca(Xtr, ytr, Xva):
    """자산 연속변수만 PCA 2축으로 축약 후 ① 방식 — '샘플링+차원축소' 코멘트 대응"""
    acols = [c for c in Xtr.columns if any(c.startswith(a) for a in ASSET_KEEP)]
    if len(acols) >= 2:
        pca = PCA(n_components=2).fit(Xtr[acols].fillna(0))
        def tf(F):
            Z = F.drop(columns=acols).copy()
            P = pca.transform(F[acols].fillna(0)); Z['ASSET_PC1'], Z['ASSET_PC2'] = P[:, 0], P[:, 1]
            return Z
        Xtr, Xva = tf(Xtr), tf(Xva)
    return fp_spw(Xtr, ytr, Xva)

EXP = {'① scale_pos_weight (현행 원칙)': fp_spw,
       '② 음성 다운샘플 1:20 × 5모델 앙상블': fp_downsample_ens,
       '④ 자산군 PCA 2축 + ①': fp_spw_pca}
if HAS_SMOTE:
    EXP['③ SMOTE (fold 내부 train만) — 대조군'] = fp_smote

# rows = []
# for name, fp in EXP.items():
#     oof = cv_oof_custom(X_exp, y_exp, groups, fp)
#     (am, alo, ahi), _ = cluster_bootstrap_ci(y_exp, oof, groups)
#     rows.append({'방법': name, 'AUROC(OOF)': round(roc_auc_score(y_exp, oof), 4),
#                  'AUROC 95% CI': f'[{alo:.3f}, {ahi:.3f}]',
#                  'AUPRC(OOF)': round(average_precision_score(y_exp, oof), 4)})
# res21 = pd.DataFrame(rows); display(res21)
# save_ckpt(res21, 'step21_불균형비교')
# print('해석 가이드: CI가 서로 겹치면 "방법 간 유의미한 차이 없음 → 가장 단순한 ① 유지"가 결론.')
# print('SMOTE의 CI가 넓거나 성능이 낮으면, 그 자체가 "합성데이터 위 합성 회피" 원칙의 실증 근거가 된다.')


## STEP 21R. 불균형 비교 공정화 — 앙상블 효과 분리 (2×2)
실행 위치: **STEP 21 이후 아무 때나** (STEP 24 다음도 무방 — 같은 런타임이면 그냥 이 셀 실행).
런타임이 새로 시작됐다면 STEP 17→21을 먼저 순차 실행해야 함 (`X_exp`, `groups`, `LGB_SMALL`, `cv_oof_custom`, `fp_spw`, `fp_downsample_ens`, `cluster_bootstrap_ci` 필요).

예상 소요: 모델 약 60회 학습 — Colab 무료 기준 수 분.

In [ ]:
# ============ STEP 21R. 불균형 비교 공정화 — "샘플링 효과"와 "앙상블 효과" 분리 (2×2 설계) ============
# 문제의식: STEP 21에서 ②(다운샘플 앙상블)가 ①(spw 단일)을 이겼지만, ②만 fold당 5개 모델 평균이라
#           이긴 이유가 '다운샘플링'인지 '앙상블'인지 구분이 안 된다.
# 설계: {가중(spw) / 다운샘플} × {단일 / 5-앙상블} 네 칸을 같은 조건(같은 fold 분할)에서 비교.
#       앙상블 다양성의 원천을 맞추기 위해 spw-앙상블에는 배깅(subsample 0.8) + seed 변화를 준다
#       (다운샘플 앙상블의 다양성이 '음성 재표집'에서 오는 것과 대응).

def fp_spw_ens(Xtr, ytr, Xva, n_models=5):
    spw = (ytr == 0).sum() / max(ytr.sum(), 1)
    preds = []
    for s in range(n_models):
        params = {**LGB_SMALL, 'random_state': 1000 + s, 'subsample': 0.8, 'subsample_freq': 1}
        m = lgb.LGBMClassifier(scale_pos_weight=spw, **params)
        m.fit(Xtr, ytr)
        preds.append(m.predict_proba(Xva)[:, 1])
    return np.mean(preds, axis=0)

def fp_down_single(Xtr, ytr, Xva):
    return fp_downsample_ens(Xtr, ytr, Xva, n_models=1)

GRID = {'① spw · 단일 (STEP 21 기준)':      fp_spw,
        '①E spw · 5-앙상블(배깅) ★공정 비교': fp_spw_ens,
        '②S 다운샘플 1:20 · 단일':           fp_down_single,
        '② 다운샘플 1:20 · 5-앙상블':         fp_downsample_ens}

rows, ci_store = [], {}
for name, fp in GRID.items():
    oof = cv_oof_custom(X_exp, y_exp, groups, fp)          # 동일 seed(42) fold 분할 → 조건 통제
    (am, alo, ahi), _ = cluster_bootstrap_ci(y_exp, oof, groups)
    ci_store[name] = (alo, ahi)
    rows.append({'구성': name, 'AUROC(OOF)': round(roc_auc_score(y_exp, oof), 4),
                 'AUROC 95% CI': f'[{alo:.3f}, {ahi:.3f}]',
                 'AUPRC(OOF)': round(average_precision_score(y_exp, oof), 4)})
res21R = pd.DataFrame(rows); display(res21R)
save_ckpt(res21R, 'step21R_불균형공정비교')

# --- 자동 판정 가이드 ---
lo_e, hi_e   = ci_store['①E spw · 5-앙상블(배깅) ★공정 비교']
lo_d, hi_d   = ci_store['② 다운샘플 1:20 · 5-앙상블']
if lo_d > hi_e:
    print('판정: ②가 ①E를 CI 비겹침으로 이김 → 다운샘플링 자체의 이득이 실재 → ② 채택.')
elif lo_e > hi_d:
    print('판정: ①E가 ② 우위 → 가중 방식 유지(앙상블만 추가) → ①E 채택.')
else:
    print('판정: ①E와 ②의 CI가 겹침 → STEP 21의 ② 우위는 대부분 "앙상블 효과"였음.')
    print('      → 방법론적으로 단순한 쪽 채택: 원칙(scale_pos_weight)을 유지하되 5-앙상블만 추가(①E).')
print('공통 확인: 단일 두 칸(①, ②S) 대비 앙상블 두 칸의 상승폭 = 앙상블 효과의 크기 — 발표에서 분해 근거로 사용.')


,구성,AUROC(OOF),AUROC 95% CI,AUPRC(OOF)
0,① spw · 단일 (STEP 21 기준),0.7132,"[0.668, 0.761]",0.0059
1,①E spw · 5-앙상블(배깅) ★공정 비교,0.7835,"[0.742, 0.820]",0.0100
2,②S 다운샘플 1:20 · 단일,0.8179,"[0.781, 0.853]",0.0105
3,② 다운샘플 1:20 · 5-앙상블,0.8208,"[0.784, 0.856]",0.0108


[ckpt 저장] step21R_불균형공정비교: 4행 → /content/drive/MyDrive/09.개인_CB정보/ckpt/step21R_불균형공정비교.parquet
판정: ①E와 ②의 CI가 겹침 → STEP 21의 ② 우위는 대부분 "앙상블 효과"였음.
      → 방법론적으로 단순한 쪽 채택: 원칙(scale_pos_weight)을 유지하되 5-앙상블만 추가(①E).
공통 확인: 단일 두 칸(①, ②S) 대비 앙상블 두 칸의 상승폭 = 앙상블 효과의 크기 — 발표에서 분해 근거로 사용.


## STEP 21T. 파라미터 조정 — 다운샘플 비율 · 트리 복잡도

**붙일 위치**: STEP 21R 셀 **바로 다음**. 같은 런타임이면 그대로 실행.
(새 런타임이면 STEP 17→21까지 순차 실행 후 → 21R → 21T)

필요 객체: `X_exp`, `y_exp`, `groups`, `LGB_SMALL`, `cv_oof_custom`, `fp_spw`, `cluster_bootstrap_ci`

**21R 재판독**: 2×2를 읽으면 다운샘플 단일(0.818) > spw 앙상블(0.784)이므로 우위의 원인은 앙상블이 아니라 다운샘플링. 21R이 자동 출력한 판정문은 앙상블끼리만 비교한 규칙이라 오류 — 무시.

예상 소요: 모델 약 (7비율+7설정+2확정) × 3seed × 5fold × 5모델 ≈ 1,200회 학습. Colab 무료 기준 10~25분. 오래 걸리면 `TUNE_SEEDS`를 2개로 줄여도 됨.

In [ ]:
# ============ STEP 21T. 파라미터 조정 — 다운샘플 비율 · 트리 복잡도 (seed 반복으로 확정) ============
# 21R 재판독: 2×2 격자를 읽으면 "다운샘플 단일(0.818) > spw 앙상블(0.784)"이므로
#             ②의 우위는 앙상블이 아니라 '음성 다운샘플링' 자체의 효과다(①vs②S CI 비겹침).
#             → 21R 셀이 자동 출력한 판정문("앙상블 효과였음")은 앙상블끼리만 비교한 규칙의 오류. 무시할 것.
#
# ★ 튜닝 과적합 경고: 양성 144건에서 여러 후보를 같은 CV로 비교하면, 이긴 후보의 점수는
#   '운 좋은 분할'을 골라낸 만큼 낙관적으로 부푼다(selection bias).
#   방어책 3가지를 코드에 내장:
#     (a) 모든 후보를 fold seed 여러 개로 평가 → 단일 분할 운을 제거
#     (b) 선택 기준을 '최고점'이 아니라 '평균 − 1×표준편차'(안정성 페널티)로
#     (c) 최종 확정 수치는 튜닝에 쓰지 않은 seed로 재측정(홀드아웃 seed)

TUNE_SEEDS = [42, 7, 2024]        # 탐색용 fold 분할 seed
HOLDOUT_SEEDS = [101, 202]        # 확정 측정 전용(탐색에 사용 금지)

def eval_multi_seed(fp, seeds, label=''):
    """여러 fold 분할 seed에서 OOF AUROC/AUPRC를 반복 측정."""
    aucs, aps = [], []
    for sd in seeds:
        oof = cv_oof_custom(X_exp, y_exp, groups, fp, seed=sd)
        aucs.append(roc_auc_score(y_exp, oof)); aps.append(average_precision_score(y_exp, oof))
    return np.mean(aucs), np.std(aucs), np.mean(aps)

def make_fp_down(ratio, n_models=5, params=None):
    """음성:양성 = ratio:1 로 다운샘플한 학습셋 n_models개의 평균 예측."""
    P = params or LGB_SMALL
    def fp(Xtr, ytr, Xva):
        pos, neg = np.where(ytr == 1)[0], np.where(ytr == 0)[0]
        preds = []
        for s in range(n_models):
            rng = np.random.default_rng(s)
            n_neg = min(len(neg), int(len(pos) * ratio))
            sel = np.concatenate([pos, rng.choice(neg, size=n_neg, replace=False)])
            m = lgb.LGBMClassifier(**{**P, 'random_state': 500 + s}); m.fit(Xtr.iloc[sel], ytr[sel])
            preds.append(m.predict_proba(Xva)[:, 1])
        return np.mean(preds, axis=0)
    return fp

# ── ① 다운샘플 비율 탐색 ──────────────────────────────────────────
print('=== ① 음성 다운샘플 비율 (트리 파라미터는 min_child=20으로 완화 고정) ===')
LGB_DOWN = {**LGB_SMALL, 'min_child_samples': 20}   # 다운샘플 학습셋 크기에 맞춘 기본값
rows = []
for ratio in [3, 5, 10, 20, 50, 100]:
    mu, sd, ap = eval_multi_seed(make_fp_down(ratio, params=LGB_DOWN), TUNE_SEEDS)
    rows.append({'비율(음성:양성)': f'{ratio}:1', 'AUROC 평균': round(mu, 4),
                 'AUROC 표준편차': round(sd, 4), '안정성점수(μ−σ)': round(mu - sd, 4),
                 'AUPRC 평균': round(ap, 4)})
mu, sd, ap = eval_multi_seed(fp_spw, TUNE_SEEDS)          # 기준선(현행 원칙)
rows.append({'비율(음성:양성)': '전체(spw 가중)', 'AUROC 평균': round(mu, 4),
             'AUROC 표준편차': round(sd, 4), '안정성점수(μ−σ)': round(mu - sd, 4),
             'AUPRC 평균': round(ap, 4)})
res_ratio = pd.DataFrame(rows); display(res_ratio)
best_row = res_ratio.iloc[res_ratio['안정성점수(μ−σ)'].idxmax()]
BEST_RATIO = None if '전체' in best_row['비율(음성:양성)'] else int(best_row['비율(음성:양성)'].split(':')[0])
print(f"→ 선택(μ−σ 기준): {best_row['비율(음성:양성)']}")
if BEST_RATIO is None:
    BEST_RATIO = 20; print('  (spw가 최고지만 이후 탐색은 다운샘플 20:1로 진행)')

# ── ② 트리 복잡도 탐색 (양성 144건 → 얕고 강한 정규화 위주로만) ──
print('\n=== ② 트리 파라미터 (비율 고정) ===')
# ★ 중요: LGB_SMALL의 min_child_samples=200은 '68,677행 전체 학습' 기준으로 정한 값이다.
#   다운샘플하면 학습셋이 (양성115 + 음성 115×ratio) 수준으로 급감하므로 200은 과도한 제약이 되어
#   트리가 분기조차 못 하는 일이 생긴다. → min_child를 다운샘플 크기에 맞춰 함께 탐색한다.
PARAM_GRID = {
    'P1 얕음(d2/leaf4)+child20':   {**LGB_SMALL, 'max_depth': 2, 'num_leaves': 4,  'min_child_samples': 20},
    'P2 현행(d3/leaf7)+child20':   {**LGB_SMALL, 'min_child_samples': 20},
    'P3 현행+child50':             {**LGB_SMALL, 'min_child_samples': 50},
    'P4 현행+child200(원본)':       {**LGB_SMALL},
    'P5 깊게(d4/leaf15)+child20':  {**LGB_SMALL, 'max_depth': 4, 'num_leaves': 15, 'min_child_samples': 20},
    'P6 낮은lr/많은트리+child20':   {**LGB_SMALL, 'learning_rate': 0.02, 'n_estimators': 800, 'min_child_samples': 20},
    'P7 강한 L2(reg=20)+child20':  {**LGB_SMALL, 'reg_lambda': 20.0, 'min_child_samples': 20},
}
rows = []
for name, P in PARAM_GRID.items():
    mu, sd, ap = eval_multi_seed(make_fp_down(BEST_RATIO, params=P), TUNE_SEEDS)
    rows.append({'설정': name, 'AUROC 평균': round(mu, 4), 'AUROC 표준편차': round(sd, 4),
                 '안정성점수(μ−σ)': round(mu - sd, 4), 'AUPRC 평균': round(ap, 4)})
res_param = pd.DataFrame(rows); display(res_param)
BEST_NAME = res_param.iloc[res_param['안정성점수(μ−σ)'].idxmax()]['설정']
BEST_PARAMS = PARAM_GRID[BEST_NAME]
print(f'→ 선택(μ−σ 기준): {BEST_NAME}')

# ── ③ 홀드아웃 seed로 확정 측정 (탐색에 쓰지 않은 분할) ──
print('\n=== ③ 확정 측정 — 홀드아웃 seed (튜닝 낙관편의 제거) ===')
fp_final = make_fp_down(BEST_RATIO, params=BEST_PARAMS)
mu_t, sd_t, ap_t = eval_multi_seed(fp_final, TUNE_SEEDS)
mu_h, sd_h, ap_h = eval_multi_seed(fp_final, HOLDOUT_SEEDS)
oof_final = cv_oof_custom(X_exp, y_exp, groups, fp_final, seed=HOLDOUT_SEEDS[0])
(am, alo, ahi), (pm, plo, phi) = cluster_bootstrap_ci(y_exp, oof_final, groups)
res_final = pd.DataFrame([{
    '최종 구성': f'다운샘플 {BEST_RATIO}:1 × 5앙상블 · {BEST_NAME}',
    '탐색 seed AUROC': f'{mu_t:.4f}±{sd_t:.4f}',
    '홀드아웃 seed AUROC': f'{mu_h:.4f}±{sd_h:.4f}',
    '낙관편의(탐색−홀드아웃)': round(mu_t - mu_h, 4),
    'AUROC 95% CI': f'[{alo:.3f}, {ahi:.3f}]',
    'AUPRC': round(ap_h, 4), '기준선': round(float(y_exp.mean()), 4)}])
display(res_final)
save_ckpt(res_ratio, 'step21T_비율탐색'); save_ckpt(res_param, 'step21T_파라미터탐색')
save_ckpt(res_final, 'step21T_최종확정')

print('\n[보고 원칙]')
print(' · 공식 성능 수치는 ③의 "홀드아웃 seed" 값을 쓴다 — 탐색 seed 값은 후보를 고르는 데 이미 사용됐다.')
print(' · 낙관편의가 0.02를 넘으면 튜닝이 분할 운을 주웠다는 신호 → 후보 수를 줄이거나 그대로 홀드아웃 값만 보고.')
print(f' · 최종 설정: ratio={BEST_RATIO}, params={BEST_NAME} → STEP 20/24 재실행 시 이 설정으로 통일.')


=== ① 음성 다운샘플 비율 (트리 파라미터는 min_child=20으로 완화 고정) ===


,비율(음성:양성),AUROC 평균,AUROC 표준편차,안정성점수(μ−σ),AUPRC 평균
0,3:1,0.8122,0.0025,0.8097,0.0106
1,5:1,0.8119,0.0012,0.8107,0.0103
2,10:1,0.8126,0.0046,0.8080,0.0110
3,20:1,0.8144,0.0055,0.8089,0.0110
4,50:1,0.8155,0.0042,0.8113,0.0116
5,100:1,0.8153,0.0040,0.8113,0.0115
6,전체(spw 가중),0.7126,0.0129,0.6997,0.0056


→ 선택(μ−σ 기준): 50:1

=== ② 트리 파라미터 (비율 고정) ===


,설정,AUROC 평균,AUROC 표준편차,안정성점수(μ−σ),AUPRC 평균
0,P1 얕음(d2/leaf4)+child20,0.8217,0.0026,0.8191,0.0119
1,P2 현행(d3/leaf7)+child20,0.8155,0.0042,0.8113,0.0116
2,P3 현행+child50,0.8154,0.0047,0.8107,0.0112
3,P4 현행+child200(원본),0.8117,0.0043,0.8074,0.0110
4,P5 깊게(d4/leaf15)+child20,0.8106,0.0036,0.8069,0.0114
5,P6 낮은lr/많은트리+child20,0.8154,0.0041,0.8113,0.0114
6,P7 강한 L2(reg=20)+child20,0.8179,0.0029,0.8150,0.0114


→ 선택(μ−σ 기준): P1 얕음(d2/leaf4)+child20

=== ③ 확정 측정 — 홀드아웃 seed (튜닝 낙관편의 제거) ===


,최종 구성,탐색 seed AUROC,홀드아웃 seed AUROC,낙관편의(탐색−홀드아웃),AUROC 95% CI,AUPRC,기준선
0,다운샘플 50:1 × 5앙상블 · P1 얕음(d2/leaf4)+child20,0.8217±0.0026,0.8198±0.0057,0.0019,"[0.788, 0.859]",0.0108,0.0021


[ckpt 저장] step21T_비율탐색: 7행 → /content/drive/MyDrive/09.개인_CB정보/ckpt/step21T_비율탐색.parquet
[ckpt 저장] step21T_파라미터탐색: 7행 → /content/drive/MyDrive/09.개인_CB정보/ckpt/step21T_파라미터탐색.parquet
[ckpt 저장] step21T_최종확정: 1행 → /content/drive/MyDrive/09.개인_CB정보/ckpt/step21T_최종확정.parquet

[보고 원칙]
 · 공식 성능 수치는 ③의 "홀드아웃 seed" 값을 쓴다 — 탐색 seed 값은 후보를 고르는 데 이미 사용됐다.
 · 낙관편의가 0.02를 넘으면 튜닝이 분할 운을 주웠다는 신호 → 후보 수를 줄이거나 그대로 홀드아웃 값만 보고.
 · 최종 설정: ratio=50, params=P1 얕음(d2/leaf4)+child20 → STEP 20/24 재실행 시 이 설정으로 통일.


## STEP 22. 2022 진입자 급증 분해

In [ ]:
# ============ STEP 22. 2022 진입자 급증(6배) 분해 — "인터넷은행발 포용 확대" 가설의 데이터 대조 ============
# 외부 근거(발표용): 금융위 「인터넷전문은행 중·저신용자 대출 확대계획」(2021.5) → 2022년말 목표
#   토스뱅크 42% / 카카오·케이뱅크 25%, 토스뱅크 2021.10 출범(첫 온기=2022), 2022 대안CSS 가동.
# 아래는 그 가설이 데이터 방향과 맞는지 확인: ①신용대출·2금융 채널 진입 비중 ②청년 비중 ③SCORE 분포.

seg  = ent_t.copy()
card = seg['C1M210000'].fillna(0) > 0
loan = seg['L10210000'].fillna(0) > 0
seg['진입상품'] = np.select([card & loan, card & ~loan, ~card & loan],
                            ['카드+대출', '카드만', '대출만'], default='기타(카드기관수 등)')
tab1 = seg.groupby(['YEAR', '진입상품']).size().unstack(fill_value=0)
print('--- ① 진입 상품 구성 (행 비율) ---'); display(tab1.div(tab1.sum(axis=1), axis=0).round(3))

loan_seg = seg[loan].copy()
loan_seg['채널'] = np.where(loan_seg['NONBANK_RATIO'] > 0, '2금융 포함', '은행만')
tab2 = loan_seg.groupby(['YEAR', '채널']).size().unstack(fill_value=0)
print('--- ② 대출 보유 진입자의 채널 (행 비율) ---'); display(tab2.div(tab2.sum(axis=1), axis=0).round(3))

print('--- ③ 인구·SCORE·불량률 ---')
display(seg.groupby('YEAR').agg(
    여성비중=('GENDER', lambda s: float((pd.to_numeric(s, errors='coerce') == 2).mean())),
    이십대이하비중=('AGE_BAND', lambda s: float((pd.to_numeric(s, errors='coerce') <= 2).mean())),
    SCORE_중앙값=('SCORE', 'median'),
    불량률_12M=('TARGET_12M', 'mean')).round(4))

print('판독: 2022에 신용대출·2금융(인터넷은행 포함) 채널과 청년 비중이 커졌다면 정책 서사와 정합.')
print('유보: 배율 6배 자체는 합성데이터 생성 특성의 증폭 가능성 병기 (Track B의 씬파일러 재고 감소와 방향 일치 = 버그 아님 근거).')


--- ① 진입 상품 구성 (행 비율) ---


진입상품,기타(카드기관수 등),대출만,카드+대출,카드만
YEAR,,,,
2020,0.057,0.123,0.011,0.808
2021,0.159,0.252,0.022,0.567
2022,0.111,0.219,0.052,0.617


--- ② 대출 보유 진입자의 채널 (행 비율) ---


채널,2금융 포함,은행만
YEAR,,
2020,0.073,0.927
2021,0.086,0.914
2022,0.122,0.878


--- ③ 인구·SCORE·불량률 ---


,여성비중,이십대이하비중,SCORE_중앙값,불량률_12M
YEAR,,,,
2020,0.3739,0.7102,807.0,0.0053
2021,0.3790,0.7476,750.0,0.0008
2022,0.3960,0.7642,752.0,0.0009


판독: 2022에 신용대출·2금융(인터넷은행 포함) 채널과 청년 비중이 커졌다면 정책 서사와 정합.
유보: 배율 6배 자체는 합성데이터 생성 특성의 증폭 가능성 병기 (Track B의 씬파일러 재고 감소와 방향 일치 = 버그 아님 근거).


## STEP 23. 연도 편중 방어

In [ ]:
# ============ STEP 23. 연도 편중 방어 — Leave-One-Year-Out + 2022 다운웨이트 민감도 ============
y_l = ent_t['TARGET_24M'].astype(int).values
X_l = X2
yr  = ent_t['YEAR'].values

print('--- ① Leave-One-Year-Out: 해당 진입연도 제외 학습 → 그 해 채점 ---')
rows = []
for hold in sorted(np.unique(yr)):
    tr, te = yr != hold, yr == hold
    if y_l[te].sum() == 0 or y_l[tr].sum() == 0:
        rows.append({'평가연도': int(hold), 'n': int(te.sum()), '양성': int(y_l[te].sum()), 'AUROC': '평가불가(양성 0)'})
        continue
    spw = (y_l[tr] == 0).sum() / max(y_l[tr].sum(), 1)
    m = lgb.LGBMClassifier(scale_pos_weight=spw, **LGB_SMALL)
    m.fit(X_l.loc[tr], y_l[tr])
    rows.append({'평가연도': int(hold), 'n': int(te.sum()), '양성': int(y_l[te].sum()),
                 'AUROC': round(roc_auc_score(y_l[te], m.predict_proba(X_l.loc[te])[:, 1]), 4)})
res_loyo = pd.DataFrame(rows); display(res_loyo)

print('--- ② 2022 표본 다운웨이트 민감도 (반복 3회 CV) ---')
rows = []
for w22 in [1.0, 0.5, 1/6]:
    sw = np.where(yr == 2022, w22, 1.0)
    rep_auc, oof = repeated_cv_oof(X_l, y_l, groups, n_repeats=3, sample_weight=sw)
    (am, alo, ahi), _ = cluster_bootstrap_ci(y_l, oof, groups)
    rows.append({'w_2022': round(w22, 3), 'AUROC(OOF)': round(roc_auc_score(y_l, oof), 4),
                 'AUROC 95% CI': f'[{alo:.3f}, {ahi:.3f}]'})
res_w = pd.DataFrame(rows); display(res_w)
save_ckpt(res_loyo, 'step23_LOYO'); save_ckpt(res_w, 'step23_다운웨이트')
print('판독: ①에서 특정 연도만 유독 성능이 다르면 연도 특성 의존 신호. ②에서 CI가 겹치면 "2022 편중이 결과를 지배하지 않는다"고 보고 가능.')


--- ① Leave-One-Year-Out: 해당 진입연도 제외 학습 → 그 해 채점 ---


,평가연도,n,양성,AUROC
0,2020,8150,48,0.6589
1,2021,8610,50,0.7203
2,2022,51917,46,0.7361


--- ② 2022 표본 다운웨이트 민감도 (반복 3회 CV) ---


,w_2022,AUROC(OOF),AUROC 95% CI
0,1.000,0.7805,"[0.739, 0.818]"
1,0.500,0.7853,"[0.745, 0.821]"
2,0.167,0.7787,"[0.739, 0.813]"


[ckpt 저장] step23_LOYO: 3행 → /content/drive/MyDrive/09.개인_CB정보/ckpt/step23_LOYO.parquet
[ckpt 저장] step23_다운웨이트: 3행 → /content/drive/MyDrive/09.개인_CB정보/ckpt/step23_다운웨이트.parquet
판독: ①에서 특정 연도만 유독 성능이 다르면 연도 특성 의존 신호. ②에서 CI가 겹치면 "2022 편중이 결과를 지배하지 않는다"고 보고 가능.


## STEP 24. 전처리 산출물 (모델링 담당 전달)

In [ ]:
# ============ STEP 24. [전처리 산출물] 구간화 CSV + WOE 유틸 — 모델링 담당 전달용 ============
# 역할 분담: 전처리는 ① 구간(bin) 정의 확정 ② woe_fit/woe_transform 함수 제공까지.
# ★ WOE 값은 타겟(y)을 쓰므로 반드시 "각 fold의 train에서만" 적합해야 한다.
#   전체 데이터로 WOE를 만들어 CSV에 박으면 그 자체가 타겟 누수다 — 그래서 CSV에는 '구간'까지만 싣는다.

def bin_series(s, n_bins=5):
    """범주형/저카디널리티는 그대로, 연속형은 0-분리 5분위. 결측은 'MISSING' 독립 범주."""
    if not pd.api.types.is_numeric_dtype(s) or s.nunique() <= 8:
        return pd.Series(np.where(s.isna(), 'MISSING', s.astype(str)), index=s.index)
    s = pd.to_numeric(s, errors='coerce')
    out = pd.Series('MISSING', index=s.index, dtype=object)
    out[s == 0] = 'zero'
    nz = s.notna() & (s != 0)
    if nz.sum() >= n_bins * 20:
        try: out[nz] = pd.qcut(s[nz], q=n_bins, duplicates='drop').astype(str)
        except ValueError: out[nz] = 'nonzero'
    elif nz.any():
        out[nz] = 'nonzero'
    return out

def woe_fit(binned, y, min_n=30):
    g = pd.DataFrame({'x': binned.values, 'y': np.asarray(y)}).groupby('x')['y'].agg(n='size', bad='sum')
    g = g[g['n'] >= min_n]
    g['good'] = g['n'] - g['bad']
    br = (g['bad'] + .5) / (g['bad'].sum() + .5)
    gr = (g['good'] + .5) / (g['good'].sum() + .5)
    return np.log(gr / br).to_dict()                 # {구간: WoE}

def woe_transform(binned, mapping):
    return binned.map(mapping).fillna(0.0)           # 미학습 구간은 0(중립)

# 모델 담당에게 전달할 사용 패턴 (fold 안에서만 적합):
#   for tr, va in sgkf.split(X, y, groups):
#       for c in FEATS:
#           b_tr, b_va = bin_series(X[c].iloc[tr]), bin_series(X[c].iloc[va])
#           mp = woe_fit(b_tr, y[tr])                     # ← train fold에서만
#           X_woe_tr[c], X_woe_va[c] = woe_transform(b_tr, mp), woe_transform(b_va, mp)

# --- 구간화 완료 v2 학습 파일 저장 (A-1 / A-2) ---
EXPORT = {
    'track_a_train_A1_binned_v2_final.csv': BASE_V2 + FLAGS + [f'DELTA_{c}' for c in AL3] + ['NONBANK_RATIO', 'GENDER', 'AGE_BAND'],
    'track_a_train_A2_binned_v2_final.csv': BASE_V2_PREV + [f'DELTA_{c}_prev' for c in AL3] + ['GENDER', 'AGE_BAND'],
}
for fname, cols in EXPORT.items():
    cols = [c for c in cols if c in ent_t.columns]
    exp = pd.DataFrame({c: bin_series(ent_t[c]) for c in cols})
    exp['TARGET_12M'] = ent_t['TARGET_12M']; exp['TARGET_24M'] = ent_t['TARGET_24M']
    exp['ID'] = ent_t['ID']; exp['ENTRY_YEAR'] = ent_t['YEAR']
    path = os.path.join(DATA_DIR, fname)
    exp.to_csv(path, index=False)
    print(f'저장: {path} ({len(exp):,}행 × {exp.shape[1]}열)')
print('전달 메모: "선형/스코어카드용은 이 구간화 파일 + woe_fit/woe_transform을 fold 내부에서 호출.'
      ' 트리용은 기존 track_a_train_*.csv(원값+플래그) 그대로." — 인수인계 요약 5절에 한 줄 추가.')


저장: /content/drive/MyDrive/09.개인_CB정보/track_a_train_A1_binned_v2_final.csv (68,677행 × 21열)
저장: /content/drive/MyDrive/09.개인_CB정보/track_a_train_A2_binned_v2_final.csv (68,677행 × 16열)
전달 메모: "선형/스코어카드용은 이 구간화 파일 + woe_fit/woe_transform을 fold 내부에서 호출. 트리용은 기존 track_a_train_*.csv(원값+플래그) 그대로." — 인수인계 요약 5절에 한 줄 추가.


# Track A — STEP 25A~28: 최종 확정 → apply v2 → H1 검증 → 산출물 갱신

**붙일 위치**: 기존 STEP 17~24 노트북의 **STEP 21T 다음**에 이 셀들을 순서대로 추가.
**실행 전제**: 같은 런타임에서 STEP 17→21이 실행돼 있어야 함(21R·21T 객체는 불필요 — 최종 설정은 하드코딩됨).
새 런타임이면: STEP 17→18→19-1~4→20→21 순차 실행 후 여기부터.

| STEP | 작업 | 산출물 |
|---|---|---|
| 25A | 최종 설정(50:1×5, depth2) 통일 적용 — 공식 4조합 표·A-2 보강 절제·기여도/방향 표 | step25_공식성능표, step25_피처기여도 |
| 25B | LOYO·다운웨이트를 최종 설정으로 재측정 | step25B_* |
| 26 | apply v2 입력 생성 (원본 2021·2022 CSV 재로드 — 유일한 예외) | apply_v2_input |
| 27 | 씬파일러 79.5만 스코어링 → H1 판정 | track_a_apply_v2_scored.csv |
| 28 | 학습파일 2종(원값 v2)·데이터사전 v2 | track_a_train_A*_v2.csv, 데이터사전_v2.csv |

In [11]:
# ============ [상태 점검] 25A 실행 전 필요한 객체 확인 ============
need = {'ent_t': 'STEP 18', 'X1': 'STEP 20', 'X2': 'STEP 20', 'groups': 'STEP 20',
        'encode': 'STEP 20', 'LGB_SMALL': 'STEP 20', 'cluster_bootstrap_ci': 'STEP 20',
        'repeated_cv_oof': 'STEP 20', 'cv_oof_custom': 'STEP 21', 'has_info': 'STEP 19-4',
        'CONF_PAIRS': 'STEP 19-1', 'ASSET_KEEP': 'STEP 19-2', 'AL3': 'STEP 19-3',
        'BASE_V2': 'STEP 20', 'BASE_V2_PREV': 'STEP 20', 'FLAGS': 'STEP 20',
        'wide_all': 'STEP 17', 'lgb': 'STEP 20', 'StratifiedGroupKFold': 'STEP 20'}
missing = [f'{k} ({v})' for k, v in need.items() if k not in dir()]
if missing:
    print('❌ 런타임이 끊겼거나 미실행 셀 있음 — 없는 객체:')
    for m in missing: print('   -', m)
    print('\n→ STEP 17 → 18 → 19-1~4 → 20 → 21 순차 실행 후 25A로.')
    print('   (21R·21T·22·23·24는 25A에 불필요 — 건너뛰어도 됨)')
else:
    print('✅ 필요한 객체 전부 존재 — 25A 바로 실행 가능')
    print(f"   진입자 {len(ent_t):,}행 | 24M 양성 {int(ent_t['TARGET_24M'].sum())}건 "
          f"| A-1 {X1.shape[1]}피처 / A-2 {X2.shape[1]}피처")

✅ 필요한 객체 전부 존재 — 25A 바로 실행 가능
   진입자 68,677행 | 24M 양성 144건 | A-1 26피처 / A-2 21피처


## STEP 25A. 최종 설정 확정 — 공식 성능표·절제·기여도
**붙일 위치**: STEP 21T 바로 다음 (같은 노트북). 필요 객체: STEP 17~21의 전부(`X1`,`X2`,`ent_t`,`groups`,`cv_oof_custom`,`has_info`,`CONF_PAIRS` 등). 21T 객체는 불필요(설정 하드코딩).

In [12]:
# ============ STEP 25A. 최종 설정 확정 — 4조합 공식 표 + A-2 보강 절제 + 기여도·방향 표 ============
# 21T 확정 설정을 전 조합에 통일 적용한다. 튜닝은 주력(A-2·24M)에서 1회만 수행했음을 문서에 명시.
RATIO_FINAL = 50
LGB_FINAL   = {**LGB_SMALL, 'max_depth': 2, 'num_leaves': 4, 'min_child_samples': 20}
TUNE_SEEDS_25    = [42, 7, 2024]     # 절제(변수 추가 판정)용
HOLDOUT_SEEDS_25 = [101, 202]        # 공식 수치 측정 전용

def fp_final_factory(ratio=RATIO_FINAL, params=None, n_models=5, collect=None):
    """다운샘플 ratio:1 × n_models 앙상블. collect 리스트를 주면 gain 중요도를 수집."""
    P = params or LGB_FINAL
    def fp(Xtr, ytr, Xva):
        pos, neg = np.where(ytr == 1)[0], np.where(ytr == 0)[0]
        preds = []
        for s in range(n_models):
            rng = np.random.default_rng(s)
            sel = np.concatenate([pos, rng.choice(neg, size=min(len(neg), int(len(pos) * ratio)), replace=False)])
            m = lgb.LGBMClassifier(**{**P, 'random_state': 500 + s})
            m.fit(Xtr.iloc[sel], ytr[sel])
            if collect is not None:
                collect.append(pd.Series(m.booster_.feature_importance('gain'), index=Xtr.columns))
            preds.append(m.predict_proba(Xva)[:, 1])
        return np.mean(preds, axis=0)
    return fp

def eval_seeds(X, y, fp, seeds):
    """여러 fold 분할 seed의 OOF AUROC. 첫 seed의 OOF로 부트스트랩 CI."""
    aucs, oof0 = [], None
    for sd in seeds:
        oof = cv_oof_custom(X, y, groups, fp, seed=sd)
        if oof0 is None: oof0 = oof
        aucs.append(roc_auc_score(y, oof))
    (am, alo, ahi), (pm, plo, phi) = cluster_bootstrap_ci(y, oof0, groups)
    return float(np.mean(aucs)), float(np.std(aucs)), (alo, ahi), float(average_precision_score(y, oof0)), oof0

# ── 25A-1. 보강 피처 생성 (A-2 관점 = 씬파일러 시절 t-1 기준) ─────────────
def _conf_ord(s):
    return s.astype(str).str.strip().str.upper().map({'A': 3, 'B': 2, 'C': 1}).fillna(0)

def _price_rank(p):
    p = pd.to_numeric(p, errors='coerce').fillna(0)
    r = pd.Series(0.0, index=p.index); nz = p > 0
    if nz.sum() >= 100:
        try: r[nz] = pd.qcut(p[nz], 5, labels=False, duplicates='drop') + 1
        except ValueError: r[nz] = 1
    elif nz.any(): r[nz] = 1
    return r

# 결합 파생 2건 (19-1 "후보" 판정분): 가격순위(0~5) × 신뢰서열(0~3)
if 'U81302010_prev' in ent_t.columns and 'U81304010_prev' in ent_t.columns:
    ent_t['COMBO_JEONSE_prev'] = _price_rank(ent_t['U81302010_prev']) * _conf_ord(ent_t['U81304010_prev'])
if 'U81201010_prev' in ent_t.columns and 'U81205010_prev' in ent_t.columns:
    ent_t['COMBO_ASSET_prev'] = _price_rank(ent_t['U81201010_prev']) * _conf_ord(ent_t['U81205010_prev'])

# CAR_FLAG·CONF 플래그의 t-1 버전 — STEP 20의 X2에는 빠져 있던 것(t 시점에만 만들었음)
if 'AL0C00001_prev' not in ent_t.columns and 'AL0C00001' in wide_all.columns:
    prv = wide_all[['ID', 'YEAR', 'AL0C00001']].copy(); prv['YEAR'] = prv['YEAR'] + 1
    ent_t = ent_t.merge(prv.rename(columns={'AL0C00001': 'AL0C00001_prev'}), on=['ID', 'YEAR'], how='left')
if 'AL0C00001_prev' in ent_t.columns:
    ent_t['CAR_FLAG_prev'] = has_info(ent_t['AL0C00001_prev'])
for conf, price in CONF_PAIRS.items():
    if f'{conf}_prev' in ent_t.columns:
        c = ent_t[f'{conf}_prev'].astype(str).str.strip().str.upper()
        ent_t[f'{price}_CONF_MISSING_prev'] = (~c.isin(['A', 'B', 'C'])).astype(int)

CONF_FLAGS_PREV = [c for c in ent_t.columns if c.endswith('_CONF_MISSING_prev')]
ADD_SETS = {
    'V0 기본(X2 그대로)': [],
    'V1 +결합파생2':      [c for c in ['COMBO_JEONSE_prev', 'COMBO_ASSET_prev'] if c in ent_t.columns],
    'V2 +차량플래그':     (['CAR_FLAG_prev'] if 'CAR_FLAG_prev' in ent_t.columns else []),
    'V3 +신뢰품질플래그3': CONF_FLAGS_PREV,
    'V4 전부':            [c for c in ['COMBO_JEONSE_prev', 'COMBO_ASSET_prev', 'CAR_FLAG_prev'] if c in ent_t.columns] + CONF_FLAGS_PREV,
}

# ── 25A-2. 절제 판정 (탐색 seed) → A-2 최종 피처셋 확정 ─────────────
y24 = ent_t['TARGET_24M'].astype(int).values
fp_f = fp_final_factory()
rows = []
for name, add in ADD_SETS.items():
    Xv = pd.concat([X2] + ([ent_t[add].astype(float)] if add else []), axis=1)
    mu, sd, ci, ap, _ = eval_seeds(Xv, y24, fp_f, TUNE_SEEDS_25)
    rows.append({'구성': name, '추가 피처 수': len(add), 'AUROC 평균': round(mu, 4),
                 '표준편차': round(sd, 4), '안정성점수(μ−σ)': round(mu - sd, 4)})
res_abl = pd.DataFrame(rows); display(res_abl)
best_abl = res_abl.iloc[res_abl['안정성점수(μ−σ)'].idxmax()]['구성']
ADD_FINAL = ADD_SETS[best_abl]
X2_FINAL  = pd.concat([X2] + ([ent_t[ADD_FINAL].astype(float)] if ADD_FINAL else []), axis=1)
print(f'→ A-2 최종 피처셋: {best_abl} (피처 {X2_FINAL.shape[1]}개)')
print('  판정 기록: 결합 파생 2건은 여기서 이기지 못하면 "IV 인플레이션 착시"로 최종 폐기 확정.')

# ── 25A-3. 공식 4조합 표 (홀드아웃 seed) ─────────────
X1_FINAL = X1
results = []
for label, X in [('A-1(진입시점)', X1_FINAL), ('A-2(진입전, 순수대안)', X2_FINAL)]:
    for tname in ['TARGET_12M', 'TARGET_24M']:
        y = ent_t[tname].astype(int).values
        mu, sd, (alo, ahi), ap, oof = eval_seeds(X, y, fp_f, HOLDOUT_SEEDS_25)
        results.append({'모델': label, '타겟': tname, '양성': int(y.sum()),
                        'AUROC(홀드아웃)': f'{mu:.4f}±{sd:.4f}',
                        'AUROC 95% CI': f'[{alo:.3f}, {ahi:.3f}]',
                        'AUPRC': round(ap, 4), '기준선': round(float(y.mean()), 4)})
        if label.startswith('A-2') and tname == 'TARGET_24M':
            OOF_A2_FINAL = oof                     # STEP 27의 H1 비교 기준으로 사용
res25 = pd.DataFrame(results); display(res25)
save_ckpt(res25, 'step25_공식성능표')

# ── 25A-4. 피처 기여도 + 방향 표 (A-2·24M 기준) — "컬럼 기여도" 피드백 답변 자료 ─────────────
imps = []
fp_imp = fp_final_factory(collect=imps)
_ = cv_oof_custom(X2_FINAL, y24, groups, fp_imp, seed=HOLDOUT_SEEDS_25[0])
imp = pd.concat(imps, axis=1).mean(axis=1)
imp = (imp / imp.sum() * 100).sort_values(ascending=False)
top = imp.head(12)
rows = []
for feat, g in top.items():
    x = X2_FINAL[feat].astype(float)
    rho = x.corr(pd.Series(y24, index=X2_FINAL.index), method='spearman')
    direction = '값↑ → 위험↑' if rho > 0 else ('값↑ → 위험↓' if rho < 0 else '무방향')
    rows.append({'피처': feat, '기여도(gain %)': round(float(g), 2),
                 '스피어만 상관': round(float(rho), 4), '방향': direction})
res_imp = pd.DataFrame(rows); display(res_imp)
save_ckpt(res_imp, 'step25_피처기여도')
print('발표 활용: 상위 피처가 생활이력·Δ·자산 계열이고 방향이 상식과 일치하면 "대안변수의 기여가 뚜렷하다"의 직접 증거.')


,구성,추가 피처 수,AUROC 평균,표준편차,안정성점수(μ−σ)
0,V0 기본(X2 그대로),0,0.8217,0.0026,0.8191
1,V1 +결합파생2,2,0.8188,0.0044,0.8144
2,V2 +차량플래그,1,0.8203,0.0024,0.8178
3,V3 +신뢰품질플래그3,3,0.8217,0.0026,0.8191
4,V4 전부,6,0.8192,0.0042,0.8150


→ A-2 최종 피처셋: V0 기본(X2 그대로) (피처 21개)
  판정 기록: 결합 파생 2건은 여기서 이기지 못하면 "IV 인플레이션 착시"로 최종 폐기 확정.


,모델,타겟,양성,AUROC(홀드아웃),AUROC 95% CI,AUPRC,기준선
0,A-1(진입시점),TARGET_12M,96,0.8870±0.0025,"[0.850, 0.913]",0.0276,0.0014
1,A-1(진입시점),TARGET_24M,144,0.8581±0.0004,"[0.826, 0.884]",0.0266,0.0021
2,"A-2(진입전, 순수대안)",TARGET_12M,96,0.8271±0.0010,"[0.783, 0.864]",0.0059,0.0014
3,"A-2(진입전, 순수대안)",TARGET_24M,144,0.8198±0.0057,"[0.788, 0.859]",0.0113,0.0021


[ckpt 저장] step25_공식성능표: 4행 → /content/drive/MyDrive/09.개인_CB정보/ckpt/step25_공식성능표.parquet


,피처,기여도(gain %),스피어만 상관,방향
0,AGE_2,26.60,-0.0413,값↑ → 위험↓
1,AL012G019_prev,17.08,0.0218,값↑ → 위험↑
2,GENDER_1,10.00,0.0237,값↑ → 위험↑
3,AL012G005_prev,9.50,0.0184,값↑ → 위험↑
4,U81301010_prev,9.46,-0.0045,값↑ → 위험↓
5,AS120G001_prev,7.47,-0.0167,값↑ → 위험↓
6,AGE_4,5.40,0.0322,값↑ → 위험↑
7,U81306010_prev,3.94,0.0129,값↑ → 위험↑
8,GENDER_2,3.19,-0.0237,값↑ → 위험↓
9,AL012G011_prev,2.06,0.0170,값↑ → 위험↑


[ckpt 저장] step25_피처기여도: 12행 → /content/drive/MyDrive/09.개인_CB정보/ckpt/step25_피처기여도.parquet
발표 활용: 상위 피처가 생활이력·Δ·자산 계열이고 방향이 상식과 일치하면 "대안변수의 기여가 뚜렷하다"의 직접 증거.


## STEP 25B. 연도 편중 방어 갱신 (최종 설정)

In [ ]:
# ============ STEP 25B. 연도 편중 방어 재실행 — 최종 설정(다운샘플 앙상블) 기준으로 갱신 ============
# STEP 23은 구설정(spw 단일)이었다. 보고서 숫자를 한 벌로 맞추기 위해 최종 설정으로 다시 잰다.
yr = ent_t['YEAR'].values

print('--- ① Leave-One-Year-Out (최종 설정) ---')
rows = []
for hold in sorted(np.unique(yr)):
    tr, te = yr != hold, yr == hold
    if y24[te].sum() == 0 or y24[tr].sum() == 0:
        rows.append({'평가연도': int(hold), 'n': int(te.sum()), '양성': int(y24[te].sum()), 'AUROC': '평가불가'}); continue
    pos, neg = np.where(y24 & tr)[0], np.where((y24 == 0) & tr)[0]
    preds = []
    for s in range(5):
        rng = np.random.default_rng(s)
        sel = np.concatenate([pos, rng.choice(neg, size=min(len(neg), int(len(pos) * RATIO_FINAL)), replace=False)])
        m = lgb.LGBMClassifier(**{**LGB_FINAL, 'random_state': 500 + s})
        m.fit(X2_FINAL.iloc[sel], y24[sel])
        preds.append(m.predict_proba(X2_FINAL.loc[te])[:, 1])
    rows.append({'평가연도': int(hold), 'n': int(te.sum()), '양성': int(y24[te].sum()),
                 'AUROC': round(roc_auc_score(y24[te], np.mean(preds, axis=0)), 4)})
res_loyo2 = pd.DataFrame(rows); display(res_loyo2)

print('--- ② 2022 다운웨이트 민감도 (최종 설정, seed 101 단일 분할) ---')
def cv_oof_weighted(X, y, groups, sw, seed=101):
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    oof = np.full(len(y), np.nan)
    for tr, va in sgkf.split(X, y, groups):
        pos, neg = tr[y[tr] == 1], tr[y[tr] == 0]
        preds = []
        for s in range(5):
            rng = np.random.default_rng(s)
            sel = np.concatenate([pos, rng.choice(neg, size=min(len(neg), int(len(pos) * RATIO_FINAL)), replace=False)])
            m = lgb.LGBMClassifier(**{**LGB_FINAL, 'random_state': 500 + s})
            m.fit(X.iloc[sel], y[sel], sample_weight=sw[sel])
            preds.append(m.predict_proba(X.iloc[va])[:, 1])
        oof[va] = np.mean(preds, axis=0)
    return oof

rows = []
for w22 in [1.0, 0.5, 1/6]:
    sw = np.where(yr == 2022, w22, 1.0)
    oof = cv_oof_weighted(X2_FINAL, y24, groups, sw)
    (am, alo, ahi), _ = cluster_bootstrap_ci(y24, oof, groups)
    rows.append({'w_2022': round(w22, 3), 'AUROC(OOF)': round(roc_auc_score(y24, oof), 4),
                 '95% CI': f'[{alo:.3f}, {ahi:.3f}]'})
res_w2 = pd.DataFrame(rows); display(res_w2)
save_ckpt(res_loyo2, 'step25B_LOYO_최종'); save_ckpt(res_w2, 'step25B_다운웨이트_최종')


--- ① Leave-One-Year-Out (최종 설정) ---


,평가연도,n,양성,AUROC
0,2020,8150,48,0.7755
1,2021,8610,50,0.7804
2,2022,51917,46,0.8212


--- ② 2022 다운웨이트 민감도 (최종 설정, seed 101 단일 분할) ---


,w_2022,AUROC(OOF),95% CI
0,1.000,0.8255,"[0.788, 0.859]"
1,0.500,0.8256,"[0.790, 0.858]"
2,0.167,0.8223,"[0.786, 0.853]"


[ckpt 저장] step25B_LOYO_최종: 3행 → /content/drive/MyDrive/09.개인_CB정보/ckpt/step25B_LOYO_최종.parquet
[ckpt 저장] step25B_다운웨이트_최종: 3행 → /content/drive/MyDrive/09.개인_CB정보/ckpt/step25B_다운웨이트_최종.parquet


## STEP 26. apply v2 생성 — 원본 CSV 재로드 (유일한 예외)
2021·2022 스냅샷 2개만, usecols로 좁혀 읽음. 파일 자동탐색이 실패하면 `PATH_2021`, `PATH_2022`를 직접 지정.

In [13]:
# ============ STEP 26. apply v2 생성 — 원본 CSV에서 필요 컬럼만 재로드 (이 노트북의 유일한 예외) ============
# 배치 스캔(STEP 12)이 '진입자 관련 행'만 걸렀기 때문에, apply 대상(2022말 씬파일러 79.5만)의
# 차량·자산·신뢰수준·AS120G001과 Δ 재료(2021년 AL012G)는 ckpt에 없다 → 원본에서 직접 읽는다.
# 필요 파일: 2021·2022 스냅샷 2개만. usecols로 좁혀 읽으므로 메모리 부담은 크지 않다.

import glob as _glob

def find_csv(year):
    pats = [f'*{year}12*.csv', f'*{year}*.csv']
    for p in pats:
        hits = sorted(_glob.glob(os.path.join(DATA_DIR, p)))
        hits = [h for h in hits if 'ckpt' not in h]
        if hits: return hits[0]
    raise FileNotFoundError(f'{year} 스냅샷 CSV를 찾지 못함 — PATH_2021/PATH_2022를 직접 지정하세요')

PATH_2021, PATH_2022 = find_csv(2021), find_csv(2022)
print('원본 경로:', PATH_2021, '|', PATH_2022)

TF_BASE5   = ['C1M210000', 'C18233003', 'C18233004', 'C18233005', 'L10220000']
TF_STRICT3 = ['C1L120004', 'L10173000', 'D10110000']
AL3        = ['AL012G005', 'AL012G011', 'AL012G019']
ASSET7     = ['U81301010', 'U81305010', 'U81306010', 'U81302010', 'U81201010', 'U81202010', 'U81102010']
CONF3      = ['U81303010', 'U81304010', 'U81205010']
STR_COLS   = CONF3 + ['AS120G001', 'AL0C00001']
COLS_2022  = ['ID'] + TF_BASE5 + TF_STRICT3 + AL3 + ASSET7 + STR_COLS + ['GENDER', 'AGE_BAND']
COLS_2021  = ['ID'] + AL3

def read_cols(path, cols):
    for enc in ('utf-8', 'utf-8-sig', 'cp949'):
        try:
            d = pd.read_csv(path, encoding=enc, usecols=lambda c: c in cols,
                            dtype={c: 'str' for c in STR_COLS if c in cols} | {'ID': 'str'})
            return d
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError('인코딩 실패', b'', 0, 0, path)

d22 = read_cols(PATH_2022, COLS_2022)
print(f'2022 로드: {len(d22):,}행 × {d22.shape[1]}열')
num_cols = [c for c in d22.columns if c not in STR_COLS + ['ID']]
d22[num_cols] = d22[num_cols].apply(pd.to_numeric, errors='coerce')

tf_base   = (d22[TF_BASE5].fillna(0) == 0).all(axis=1)
tf_strict = tf_base & (d22[TF_STRICT3].fillna(0) == 0).all(axis=1)
apply_df  = d22[tf_base].copy()
apply_df['IS_STRICT'] = tf_strict[tf_base].values
print(f'2022말 씬파일러(기본): {len(apply_df):,}명 | 엄격: {int(apply_df.IS_STRICT.sum()):,}명'
      '  ← 기존 794,773 / 333,571과 대조해 검증')

d21 = read_cols(PATH_2021, COLS_2021)
d21[AL3] = d21[AL3].apply(pd.to_numeric, errors='coerce')
apply_df = apply_df.merge(d21.rename(columns={c: f'{c}_2021' for c in AL3}), on='ID', how='left')
del d22, d21; gc.collect()

# ── 모델 입력 규격으로 재명명: A-2의 "_prev(씬파일러 시절)" = apply에서는 2022년 현재 ──
out = pd.DataFrame({'ID': apply_df['ID'], 'IS_STRICT': apply_df['IS_STRICT'],
                    'GENDER': apply_df['GENDER'], 'AGE_BAND': apply_df['AGE_BAND']})
for c in AL3 + ASSET7 + ['AS120G001'] + CONF3 + ['AL0C00001']:
    out[f'{c}_prev'] = apply_df[c]
for c in AL3:   # Δ = 2022 − 2021 (진입자의 DELTA_*_prev와 동일 정의)
    out[f'DELTA_{c}_prev'] = (pd.to_numeric(apply_df[c], errors='coerce')
                              - pd.to_numeric(apply_df[f'{c}_2021'], errors='coerce')).clip(lower=0)
out['CAR_FLAG_prev'] = has_info(out['AL0C00001_prev'])
for conf, price in CONF_PAIRS.items():
    c = out[f'{conf}_prev'].astype(str).str.strip().str.upper()
    out[f'{price}_CONF_MISSING_prev'] = (~c.isin(['A', 'B', 'C'])).astype(int)
if 'COMBO_JEONSE_prev' in X2_FINAL.columns:
    out['COMBO_JEONSE_prev'] = _price_rank(out['U81302010_prev']) * _conf_ord(out['U81304010_prev'])
if 'COMBO_ASSET_prev' in X2_FINAL.columns:
    out['COMBO_ASSET_prev'] = _price_rank(out['U81201010_prev']) * _conf_ord(out['U81205010_prev'])

save_ckpt(out, 'apply_v2_input')
print(f'apply v2 입력: {out.shape[0]:,}행 × {out.shape[1]}열 — STEP 27에서 인코딩·스코어링')


원본 경로: /content/drive/MyDrive/09.개인_CB정보/202112_개인CB.csv | /content/drive/MyDrive/09.개인_CB정보/202212_개인CB.csv
2022 로드: 3,129,036행 × 26열
2022말 씬파일러(기본): 794,773명 | 엄격: 333,571명  ← 기존 794,773 / 333,571과 대조해 검증
[ckpt 저장] apply_v2_input: 794,773행 → /content/drive/MyDrive/09.개인_CB정보/ckpt/apply_v2_input.parquet
apply v2 입력: 794,773행 × 26열 — STEP 27에서 인코딩·스코어링


## STEP 27. apply 스코어링 → H1 검증 (결론부)

In [ ]:
# ============ STEP 27. apply 스코어링 → H1 검증 (프로젝트 결론부) ============
# A-2 최종 모델을 2022말 씬파일러 전체에 투입한다. 이들에겐 정답이 없으므로 "채점"이 아니라
# "점수 분포"를 본다: 저위험 쏠림이면 H1("씬파일러는 위험군이 아니라 정보부족군") 지지.

apply_in = load_ckpt('apply_v2_input')

# ── 학습(진입자 전체)으로 최종 앙상블 적합 ──
pos, neg = np.where(y24 == 1)[0], np.where(y24 == 0)[0]
models = []
for s in range(5):
    rng = np.random.default_rng(s)
    sel = np.concatenate([pos, rng.choice(neg, size=min(len(neg), int(len(pos) * RATIO_FINAL)), replace=False)])
    m = lgb.LGBMClassifier(**{**LGB_FINAL, 'random_state': 500 + s})
    m.fit(X2_FINAL.iloc[sel], y24[sel]); models.append(m)

# ── apply 인코딩: 학습과 동일 규칙 → 학습 시 존재한 더미 컬럼에 정렬(없는 더미는 0) ──
raw_cols = [c for c in X2_FINAL.columns if c in apply_in.columns]        # 수치형 그대로 대응되는 것
cat_srcs = ['AS120G001_prev', 'GENDER', 'AGE_BAND']                       # 더미로 갈라졌던 원천
Xa = encode(apply_in, [c for c in apply_in.columns
                       if c.endswith('_prev') or c in ('GENDER', 'AGE_BAND')])
Xa = Xa.reindex(columns=X2_FINAL.columns, fill_value=0)
score = np.mean([m.predict_proba(Xa)[:, 1] for m in models], axis=0)
apply_in['SCORE_A2'] = score
print(f'스코어링 완료: {len(score):,}명')

# ── H1 검증 ① 진입자 위험분포 대비 apply 분포의 위치 ──
ent_q = np.quantile(OOF_A2_FINAL, np.arange(0.1, 1.0, 0.1))              # 진입자 OOF 십분위 경계
band = np.digitize(score, ent_q)                                          # 0=진입자 최저위험 10분위
dist = pd.Series(band).value_counts(normalize=True).sort_index()
tbl = pd.DataFrame({'apply 비중': (dist * 100).round(2)})
tbl.index = [f'진입자 위험 {i+1}분위 구간' for i in tbl.index]
display(tbl)
low3 = float((band <= 2).mean() * 100)
print(f'진입자 기준 하위 3분위(저위험) 구간에 속한 apply 비중: {low3:.1f}%  (균등이면 30%)')

# ── H1 검증 ② 엄격 vs 비엄격 씬파일러 ──
grp = apply_in.groupby('IS_STRICT')['SCORE_A2'].agg(['mean', 'median', lambda s: np.quantile(s, 0.9)])
grp.columns = ['평균', '중앙값', 'P90']; grp.index = grp.index.map({True: '엄격', False: '비엄격(폐계좌 등)'})
display(grp.round(5))

# ── H1 검증 ③ 진입자 실측과의 삼각측량 ──
print(f'참조: 진입자 24M 실측 불량률 {y24.mean():.4%} | apply 평균 예측점수 {score.mean():.5f} | 진입자 OOF 평균 {OOF_A2_FINAL.mean():.5f}')
print('판정 가이드:')
print(' · ①에서 apply가 저위험 구간에 과대 분포(하위 3분위 비중 > 40~50%)하고,')
print(' · ②에서 엄격 씬파일러가 비엄격보다 낮은 점수를 보이면')
print(' → H1 지지: "무이력자는 대안정보 기준으로도 대체로 저위험 — 정보 부족 때문에 배제된 집단"')
print(' · Track B의 씬파일러 스코어링 분포(LR 저위험 쏠림)와 방향 일치 여부를 결론 슬라이드에 병기.')

scored = apply_in[['ID', 'IS_STRICT', 'SCORE_A2']]
save_ckpt(scored, 'apply_v2_scored')
scored.to_csv(os.path.join(DATA_DIR, 'track_a_apply_v2_scored.csv'), index=False)
print('저장: track_a_apply_v2_scored.csv (+ ckpt apply_v2_scored)')


[ckpt 로드] apply_v2_input: 794,773행
스코어링 완료: 794,773명


,apply 비중
진입자 위험 3분위 구간,0.00
진입자 위험 4분위 구간,0.03
진입자 위험 5분위 구간,1.91
진입자 위험 6분위 구간,4.25
진입자 위험 7분위 구간,21.78
진입자 위험 8분위 구간,48.70
진입자 위험 9분위 구간,21.87
진입자 위험 10분위 구간,1.46


진입자 기준 하위 3분위(저위험) 구간에 속한 apply 비중: 0.0%  (균등이면 30%)


,평균,중앙값,P90
IS_STRICT,,,
비엄격(폐계좌 등),0.01954,0.01719,0.02962
엄격,0.01923,0.01734,0.02931


참조: 진입자 24M 실측 불량률 0.2097% | apply 평균 예측점수 0.01941 | 진입자 OOF 평균 0.01857
판정 가이드:
 · ①에서 apply가 저위험 구간에 과대 분포(하위 3분위 비중 > 40~50%)하고,
 · ②에서 엄격 씬파일러가 비엄격보다 낮은 점수를 보이면
 → H1 지지: "무이력자는 대안정보 기준으로도 대체로 저위험 — 정보 부족 때문에 배제된 집단"
 · Track B의 씬파일러 스코어링 분포(LR 저위험 쏠림)와 방향 일치 여부를 결론 슬라이드에 병기.
[ckpt 저장] apply_v2_scored: 794,773행 → /content/drive/MyDrive/09.개인_CB정보/ckpt/apply_v2_scored.parquet
저장: track_a_apply_v2_scored.csv (+ ckpt apply_v2_scored)


## STEP 27R. 스코어링 정합성 진단 + 재스코어링 (H1 확정 전 필수)

**붙일 위치**: STEP 27 다음. 같은 런타임에서 실행 (25A의 `X2_FINAL`·`eval_seeds`·`fp_final_factory`,
STEP 20의 `encode`·`BASE_V2_PREV` 필요).

**왜 필요한가**: 27의 십분위 분포(하위 3분위 0%, 7~9분위 92%)는 학습(parquet)과 apply(CSV)의
dtype 차이로 더미 컬럼명이 어긋나(`GENDER_1` vs `GENDER_1.0` 등) 인구·평형 피처가 전부 0으로
죽었을 가능성이 높다 — 기여도 45%가 인구변수였으므로 결과 전체가 왜곡된다.
이 셀은 ①죽은 피처를 진단하고 ②학습·apply를 결합 인코딩으로 재정렬해 재스코어링한 뒤 ③H1 표를 다시 만든다.

In [14]:
# ============ STEP 27R. 스코어링 정합성 진단 + 재스코어링 — H1 확정 전 필수 검증 ============
# 27의 십분위 분포가 극단(하위 3분위 0.0%, 7~9분위에 92%)으로 나왔다.
# 이 정도 극단값은 '발견'이기 전에 '버그'를 먼저 의심해야 한다.
# 유력 용의자: 학습(ent_t, parquet 유래)과 apply(원본 CSV 유래)의 dtype 차이.
#   - 학습에서 GENDER=1(정수) → 더미 'GENDER_1' / apply에서 1.0(실수) → 'GENDER_1.0'
#   - 학습에서 AS120G001_prev가 숫자코드 → 수치형 유지 / apply에서 문자로 읽힘 → 더미로 분해
#   → reindex(fill_value=0)가 이름이 다른 컬럼을 전부 0으로 채워 인구·평형 피처가 '사망'했을 가능성.
#   기여도 표에서 인구(AGE·GENDER)가 45%를 차지했으므로, 이들이 죽으면 점수가 통째로 왜곡된다.

# ── ① 진단: 학습 대비 apply에서 '죽은' 피처 탐지 ──
apply_in = load_ckpt('apply_v2_input')
Xa_old = encode(apply_in, [c for c in apply_in.columns if c.endswith('_prev') or c in ('GENDER', 'AGE_BAND')])
Xa_old = Xa_old.reindex(columns=X2_FINAL.columns, fill_value=0)
diag = pd.DataFrame({'학습 평균': X2_FINAL.mean(), 'apply 평균': Xa_old.mean()})
diag['사망 의심'] = (diag['apply 평균'] == 0) & (diag['학습 평균'].abs() > 1e-9)
dead = diag[diag['사망 의심']]
if len(dead): display(dead.round(4))
n_dead = int(diag['사망 의심'].sum())
print(f'→ 사망 피처 {n_dead}개'
      + (' — STEP 27의 분포는 이 상태로 계산된 것이므로 신뢰 불가. 아래 재스코어링으로 대체.' if n_dead
         else ' — 인코딩 정상. 27 결과가 유효하며, 고위험 쏠림은 실제 신호로 해석 대상.'))

# ── ② 결합 인코딩 재스코어링 — 학습·apply를 한 틀에서 인코딩해 어긋남을 원천 차단 ──
RAW_A2 = BASE_V2_PREV + [f'DELTA_{c}_prev' for c in AL3]
tr = ent_t[RAW_A2 + ['GENDER', 'AGE_BAND']].copy(); tr['_SET'] = 'TR'
ap = apply_in[RAW_A2 + ['GENDER', 'AGE_BAND']].copy(); ap['_SET'] = 'AP'
both = pd.concat([tr, ap], ignore_index=True)
for c in RAW_A2 + ['GENDER', 'AGE_BAND']:                 # dtype 정합: 학습 쪽 dtype 기준
    if pd.api.types.is_numeric_dtype(ent_t[c]):
        both[c] = pd.to_numeric(both[c], errors='coerce')
    else:
        both[c] = both[c].astype(str)
XJ = encode(both, RAW_A2)
g = pd.to_numeric(both['GENDER'], errors='coerce').astype('Int64')
a = pd.to_numeric(both['AGE_BAND'], errors='coerce').astype('Int64')
XJ = pd.concat([XJ, pd.get_dummies(g, prefix='GENDER'), pd.get_dummies(a, prefix='AGE')], axis=1)
XJ = XJ.astype(float)
is_tr = (both['_SET'] == 'TR').values
X_tr_j = XJ[is_tr].reset_index(drop=True)
X_ap_j = XJ[~is_tr].reset_index(drop=True)
print(f'결합 인코딩 완료: 학습 {X_tr_j.shape} | apply {X_ap_j.shape} (컬럼 동일 보장)')

# 정합성 확인 1: 재인코딩 학습행렬의 홀드아웃 성능이 기존(0.8198)과 유사해야 정상
mu, sd, (alo, ahi), ap_s, oof_j = eval_seeds(X_tr_j, y24, fp_final_factory(), HOLDOUT_SEEDS_25)
print(f'재인코딩 학습 성능: AUROC {mu:.4f}±{sd:.4f} [{alo:.3f}, {ahi:.3f}]  ← 25A의 0.8198±와 대조')

# 최종 앙상블 재적합 → apply 재스코어링
pos, neg = np.where(y24 == 1)[0], np.where(y24 == 0)[0]
models_j = []
for s in range(5):
    rng = np.random.default_rng(s)
    sel = np.concatenate([pos, rng.choice(neg, size=min(len(neg), int(len(pos) * RATIO_FINAL)), replace=False)])
    m = lgb.LGBMClassifier(**{**LGB_FINAL, 'random_state': 500 + s})
    m.fit(X_tr_j.iloc[sel], y24[sel]); models_j.append(m)
score2 = np.mean([m.predict_proba(X_ap_j)[:, 1] for m in models_j], axis=0)
apply_in['SCORE_A2_R'] = score2

# ── ③ H1 표 재생성 ──
ent_q = np.quantile(oof_j, np.arange(0.1, 1.0, 0.1))
band = np.digitize(score2, ent_q)
dist = pd.Series(band).value_counts(normalize=True).sort_index()
tbl = pd.DataFrame({'apply 비중(%)': (dist * 100).round(2)})
tbl.index = [f'진입자 위험 {i+1}분위' for i in tbl.index]
display(tbl)
low3 = float((band <= 2).mean() * 100)
print(f'하위 3분위(저위험) apply 비중: {low3:.1f}%  (균등이면 30%)')
grp = apply_in.groupby('IS_STRICT')['SCORE_A2_R'].agg(['mean', 'median'])
grp.index = grp.index.map({True: '엄격', False: '비엄격(폐계좌 등)'})
display(grp.round(5))

# ── ④ 남는 분포 차이의 '조성' 설명 — 진입자 vs 잔류자 인구 비교 ──
comp = pd.DataFrame({
    '진입자(학습)': [float((pd.to_numeric(ent_t['AGE_BAND'], errors='coerce') <= 2).mean()),
                   float((pd.to_numeric(ent_t['GENDER'], errors='coerce') == 1).mean())],
    'apply(잔류 씬파일러)': [float((pd.to_numeric(apply_in['AGE_BAND'], errors='coerce') <= 2).mean()),
                          float((pd.to_numeric(apply_in['GENDER'], errors='coerce') == 1).mean())]},
    index=['20대이하 비중', '남성 비중']).round(3)
display(comp)

scored = apply_in[['ID', 'IS_STRICT', 'SCORE_A2_R']].rename(columns={'SCORE_A2_R': 'SCORE_A2'})
save_ckpt(scored, 'apply_v2_scored_R')
scored.to_csv(os.path.join(DATA_DIR, 'track_a_apply_v2_scored_R.csv'), index=False)

print('\n[판독 가이드]')
print(' · ①에서 사망 피처가 있었다면: 27 결과는 폐기, 이 셀의 ③이 공식 H1 결과.')
print(' · 재스코어링 후 저위험 쏠림(하위 3분위 ≫ 30%)이면: H1 지지로 보고.')
print(' · 재스코어링 후에도 고위험 쏠림이 남으면: 버그가 아니라 "진입자 vs 잔류자"의 실제 차이다.')
print('   그 경우 ④의 조성 차이(연령·성별)가 설명 변수인지 확인하고, H1 문구를')
print('   "잔류 씬파일러는 대안정보 기준 진입자보다 높은 위험 신호를 보인다 — 다만 절대 예측확률은')
print('    미보정이며 조성(연령) 차이가 상당 부분을 설명" 방향으로 재구성해야 한다.')


[ckpt 로드] apply_v2_input: 794,773행


,학습 평균,apply 평균,사망 의심
AS120G001_prev,-3.4957,0.0,True
GENDER_1,0.6088,0.0,True
GENDER_2,0.3912,0.0,True
AGE_1,0.0005,0.0,True
AGE_2,0.7553,0.0,True
AGE_3,0.0860,0.0,True
AGE_4,0.0610,0.0,True
AGE_5,0.0470,0.0,True
AGE_6,0.0229,0.0,True
AGE_7,0.0193,0.0,True


→ 사망 피처 12개 — STEP 27의 분포는 이 상태로 계산된 것이므로 신뢰 불가. 아래 재스코어링으로 대체.
결합 인코딩 완료: 학습 (68677, 21) | apply (794773, 21) (컬럼 동일 보장)
재인코딩 학습 성능: AUROC 0.8198±0.0057 [0.788, 0.859]  ← 25A의 0.8198±와 대조


,apply 비중(%)
진입자 위험 1분위,5.43
진입자 위험 2분위,5.16
진입자 위험 3분위,6.39
진입자 위험 4분위,5.82
진입자 위험 5분위,6.58
진입자 위험 6분위,9.90
진입자 위험 7분위,16.03
진입자 위험 8분위,11.95
진입자 위험 9분위,18.43
진입자 위험 10분위,14.31


하위 3분위(저위험) apply 비중: 17.0%  (균등이면 30%)


,mean,median
IS_STRICT,,
비엄격(폐계좌 등),0.02979,0.01472
엄격,0.01936,0.00972


,진입자(학습),apply(잔류 씬파일러)
20대이하 비중,0.756,0.357
남성 비중,0.609,0.486


[ckpt 저장] apply_v2_scored_R: 794,773행 → /content/drive/MyDrive/09.개인_CB정보/ckpt/apply_v2_scored_R.parquet

[판독 가이드]
 · ①에서 사망 피처가 있었다면: 27 결과는 폐기, 이 셀의 ③이 공식 H1 결과.
 · 재스코어링 후 저위험 쏠림(하위 3분위 ≫ 30%)이면: H1 지지로 보고.
 · 재스코어링 후에도 고위험 쏠림이 남으면: 버그가 아니라 "진입자 vs 잔류자"의 실제 차이다.
   그 경우 ④의 조성 차이(연령·성별)가 설명 변수인지 확인하고, H1 문구를
   "잔류 씬파일러는 대안정보 기준 진입자보다 높은 위험 신호를 보인다 — 다만 절대 예측확률은
    미보정이며 조성(연령) 차이가 상당 부분을 설명" 방향으로 재구성해야 한다.


## STEP 27F2 · 29F2 — F2 기준 재스코어링 + 연령 층화 재검증

### 왜 필요한가
공식 성능 수치는 **F2(인구 전면 배제, AUROC 0.679)**로 확정했으나, `SCORE_A2`(apply 점수)와
STEP 29 연령 층화 분석은 **F0(인구 포함)** 모델로 생성돼 있다.

- 27R 실행 로그의 `결합 인코딩 완료: 학습 (68677, 21)` → 21열 = 공식 피처 10 + 인구 더미 11 (**F0 확정**)
- 27R 자가검증 수치 `AUROC 0.8198` = A-2 24M **F0** 값과 일치 (F2였다면 0.679)

문제는 **연령 층화 분석의 논리**다. "연령 구성이 교란요인"이라는 결론을 *연령을 직접 쓴 모델*의
점수로 입증한 구조라 동어반복에 가깝다. 연령을 넣었으니 연령 효과가 나오는 게 당연하고,
그걸 표준화로 걷어내면 사라지는 것도 당연하다.

### F2로 하면 무엇이 달라지나
`AGE_BAND` 컬럼은 데이터에 그대로 있으므로, **모델 입력에서만 빼고 사후 층화 분석은 그대로 가능**하다.

- **패턴이 유지되면** → "연령을 모델에 쓰지 않아도 대리변수(거주지 시세·주소 이동 등)를 통해
  연령 효과가 들어온다"는 **더 강한 발견**. 공정성 관점에서도 의미가 크다.
- **패턴이 사라지면** → 기존 관찰이 연령을 직접 넣은 결과였다는 뜻. 발표 전에 반드시 알아야 할 사실.

---

### 붙일 위치
**STEP 27R 셀 바로 다음, STEP 29보다 앞.** (기존 27R·29 셀은 지우지 말고 그대로 둘 것 — F0 기준 기록으로 보존)

### 필요 객체 (전부 27R까지 실행하면 준비됨)
`X_tr_j`, `X_ap_j`, `apply_in`, `ent_t`, `y24`, `eval_seeds`, `fp_final_factory`,
`HOLDOUT_SEEDS_25`, `RATIO_FINAL`, `LGB_FINAL`, `save_ckpt`, `DATA_DIR`

### 실행 순서
- **런타임이 살아 있으면**(27R까지 돌린 세션): 이 셀 하나만 실행. 3~8분.
- **새 런타임이면**: `17 → 18 → 19-1~4 → 20 → 21 → 25A → 26 → 27R → 이 셀`
  - 21R·21T·22·23·24·27·29·30·30R은 건너뛰어도 됨
  - 시간 절약: STEP 20은 `results = []`부터, STEP 21은 `rows = []`부터 셀 끝까지 주석 처리하면 함수 정의만 남아 빠르게 끝남

### 결과 반영
- 산출 파일: `track_a_apply_v2_scored_F2.csv`
- **F2를 공식으로 채택하면** 이 파일이 `track_a_apply_v2_scored_R.csv`를 대체하고,
  `03_전달파일_7종_설명.md`의 파일 5 설명과 STEP 29 결과 수치를 갱신해야 함
- **어느 쪽을 쓰든 문서에 생성 기준(F0/F2)을 명시**할 것 — 지금은 "A-2 최종 모델"로만 적혀 있어 구분이 안 됨


In [17]:
import inspect
src = inspect.getsource(eval_seeds)
print(src[:1200])

def eval_seeds(X, y, fp, seeds):
    """여러 fold 분할 seed의 OOF AUROC. 첫 seed의 OOF로 부트스트랩 CI."""
    aucs, oof0 = [], None
    for sd in seeds:
        oof = cv_oof_custom(X, y, groups, fp, seed=sd)
        if oof0 is None: oof0 = oof
        aucs.append(roc_auc_score(y, oof))
    (am, alo, ahi), (pm, plo, phi) = cluster_bootstrap_ci(y, oof0, groups)
    return float(np.mean(aucs)), float(np.std(aucs)), (alo, ahi), float(average_precision_score(y, oof0)), oof0



In [16]:
# ============ STEP 27F2 · 29F2. F2(인구 전면 배제) 기준 재스코어링 + 연령 층화 재검증 ============
# 배경: 공식 성능 수치는 F2(인구 제외, AUROC 0.679)로 확정했으나,
#       apply 점수(SCORE_A2)와 STEP 29 연령 층화 분석은 F0(인구 포함) 모델로 생성돼 있었다.
#       → 기준이 어긋나 있고, 특히 "연령 구성이 교란요인"이라는 결론을
#         '연령을 직접 쓴 모델'의 점수로 입증한 구조라 논리가 약하다(동어반복).
#
# 이 셀이 하는 일:
#   ① F2 피처셋(인구 더미 전면 제외)으로 재적합 → apply 재스코어링
#   ② 그 F2 점수 위에서 STEP 29(연령 층화 + 연령 표준화)를 그대로 재수행
#
# ★ 핵심: 연령을 '모델 입력'에서 뺐을 뿐, AGE_BAND 컬럼은 데이터에 그대로 있으므로
#   점수를 낸 뒤 연령별로 묶어 비교하는 사후 분석은 문제없이 가능하다.
#   오히려 F2에서 패턴이 유지되면 "연령을 안 써도 대리변수를 통해 연령 효과가 남는다"는
#   훨씬 강한 발견이 된다.

from sklearn.metrics import roc_auc_score

# ── ① F2 피처셋 구성 (27R의 결합 인코딩 결과에서 인구 더미만 제거) ──
DEMO_COLS = [c for c in X_tr_j.columns if c.startswith(('GENDER', 'AGE'))]
X_tr_f2 = X_tr_j.drop(columns=DEMO_COLS)
X_ap_f2 = X_ap_j.drop(columns=DEMO_COLS)
assert list(X_tr_f2.columns) == list(X_ap_f2.columns), '학습/apply 컬럼 불일치'
print(f'F2 피처셋: {X_tr_f2.shape[1]}개 (인구 더미 {len(DEMO_COLS)}개 제거) | '
      f'학습 {X_tr_f2.shape} · apply {X_ap_f2.shape}')

# 정합성 확인: F2 학습 성능이 기존 확정치(0.679)와 유사해야 정상
mu, sd, (am, alo, ahi), _, oof_f2 = eval_seeds(X_tr_f2, y24, fp_final_factory(), HOLDOUT_SEEDS_25)
print(f'F2 학습 성능: AUROC {mu:.4f}±{sd:.4f} [{alo:.3f}, {ahi:.3f}]  ← STEP 30의 0.679 [0.638, 0.721]와 대조')

# ── ② F2 앙상블 재적합 → apply 재스코어링 ──
pos, neg = np.where(y24 == 1)[0], np.where(y24 == 0)[0]
models_f2 = []
for s in range(5):
    r = np.random.default_rng(s)
    sel = np.concatenate([pos, r.choice(neg, size=min(len(neg), int(len(pos) * RATIO_FINAL)), replace=False)])
    m = lgb.LGBMClassifier(**{**LGB_FINAL, 'random_state': 500 + s})
    m.fit(X_tr_f2.iloc[sel], y24[sel]); models_f2.append(m)
score_f2 = np.mean([m.predict_proba(X_ap_f2)[:, 1] for m in models_f2], axis=0)
apply_in['SCORE_A2_F2'] = score_f2
print(f'F2 스코어링 완료: {len(score_f2):,}명')

# ── ③ H1 표 (F2 기준) ──
q_f2 = np.quantile(oof_f2, np.arange(0.1, 1.0, 0.1))
band_f2 = np.digitize(score_f2, q_f2)
dist = pd.Series(band_f2).value_counts(normalize=True).sort_index() * 100
tbl = pd.DataFrame({'apply 비중(%)': dist.round(2)})
tbl.index = [f'진입자 위험 {i+1}분위' for i in tbl.index]
display(tbl)
low3_raw = float((band_f2 <= 2).mean() * 100)
print(f'하위 3분위(저위험) 비중: {low3_raw:.1f}%  (균등이면 30%)')

grp = apply_in.groupby('IS_STRICT')['SCORE_A2_F2'].agg(['mean', 'median'])
grp.index = grp.index.map({True: '엄격(순수 무이력)', False: '비엄격(폐계좌 등)'})
display(grp.round(5))

# ── ④ 연령 층화 (STEP 29를 F2 점수로 재수행) ──
print('\n=== ④ 같은 연령대 안에서 진입자 vs 잔류자 (F2 점수) ===')
ent_age = pd.to_numeric(ent_t['AGE_BAND'], errors='coerce')
app_age = pd.to_numeric(apply_in['AGE_BAND'], errors='coerce')
oof_s, app_s = pd.Series(oof_f2), apply_in['SCORE_A2_F2']

rows = []
for b in sorted(ent_age.dropna().unique()):
    e = oof_s[(ent_age == b).values]; a = app_s[(app_age == b).values]
    if len(e) < 100 or len(a) < 100: continue
    rows.append({'연령대': int(b), '진입자 n': len(e), '잔류자 n': len(a),
                 '진입자 중앙값': round(float(e.median()), 5),
                 '잔류자 중앙값': round(float(a.median()), 5),
                 '배율(잔류/진입)': round(float(a.median() / max(e.median(), 1e-9)), 2)})
res_age_f2 = pd.DataFrame(rows); display(res_age_f2)
save_ckpt(res_age_f2, 'step29F2_연령층화')

print('=== ⑤ 연령 표준화 (잔류자를 진입자 연령 구성으로 재가중) ===')
ent_comp, app_comp = ent_age.value_counts(normalize=True), app_age.value_counts(normalize=True)
w = app_age.map(ent_comp / app_comp).fillna(0).values
wd = pd.Series(w).groupby(pd.Series(band_f2)).sum(); wd = wd / wd.sum() * 100
raw = pd.Series(band_f2).value_counts(normalize=True).sort_index() * 100
cmp_tbl = pd.DataFrame({'원분포(%)': raw.round(2), '연령표준화 후(%)': wd.round(2)})
cmp_tbl.index = [f'진입자 위험 {i+1}분위' for i in cmp_tbl.index]
display(cmp_tbl)
low3_std = float(wd.reindex([0, 1, 2]).fillna(0).sum())
print(f'하위 3분위 비중: 원 {low3_raw:.1f}% → 연령 표준화 후 {low3_std:.1f}%  (균등 30%)')

# ── ⑥ 산출물 저장 ──
scored_f2 = apply_in[['ID', 'IS_STRICT', 'SCORE_A2_F2']].rename(columns={'SCORE_A2_F2': 'SCORE_A2'})
save_ckpt(scored_f2, 'apply_v2_scored_F2')
scored_f2.to_csv(os.path.join(DATA_DIR, 'track_a_apply_v2_scored_F2.csv'), index=False)
print(f'\n저장: track_a_apply_v2_scored_F2.csv ({len(scored_f2):,}행)')

print('\n[판독 가이드]')
print(' · F2 학습 성능이 0.679 근처면 재현 정상. 크게 다르면 27R의 결합 인코딩부터 재확인.')
print(' · ④의 배율이 전 연령대에서 1.0 미만이고 ⑤에서 표준화 후 하위 3분위가 올라가면')
print('   → F0에서 본 패턴이 F2에서도 재현된 것. "연령을 모델에 쓰지 않아도 연령 구성이')
print('     분포를 좌우한다"는 더 강한 결론으로 승격 가능(대리변수 경로).')
print(' · 패턴이 사라지면 → 기존 관찰은 연령을 직접 넣었기 때문이었다는 뜻.')
print('   이 경우 발표에서 연령 교란 주장을 철회하고, F0 기준 관찰이었음을 명시해야 한다.')
print(' · 어느 쪽이든 문서에 SCORE_A2의 생성 기준(F0/F2)을 반드시 명시할 것.')


F2 피처셋: 10개 (인구 더미 11개 제거) | 학습 (68677, 10) · apply (794773, 10)


ValueError: not enough values to unpack (expected 3, got 2)

## STEP 29·30 — 마지막 분석 두 개

**붙일 위치**: STEP 27R 바로 다음, 순서대로. 같은 런타임 필수 — 27R의 `apply_in(SCORE_A2_R)`, `oof_j`, `X_tr_j`와 25A의 `eval_seeds`, `fp_final_factory`를 사용.

- **STEP 29 (연령 층화)**: 잔류 씬파일러의 상위 쏠림이 연령 조성 때문인지 판정 — ① 같은 연령대끼리 중앙값 비교 ② 진입자 연령 조성으로 표준화 후 십분위 재계산. 몇 초 수준.
- **STEP 30 (인구 제외 민감도)**: F0 전체 / F1 성별 제외 / F2 인구 전부 제외(순수 대안변수) 3구성 홀드아웃 비교 + F2 기여도. 홀드아웃 2seed × 3구성이라 몇 분 수준.

In [ ]:
# ============ STEP 29. 연령 층화 비교 — "잔류자 고위험 쏠림"이 연령 조성 때문인지 판정 ============
# 27R에서 남은 질문: apply(잔류 씬파일러)의 상위 쏠림이 행동 신호인가, 연령 조성(20대 76% vs 36%)인가.
# ① 같은 연령대끼리 비교 ② 연령 표준화(진입자 조성으로 가중) 후 분포 재계산 — 두 방식으로 답한다.
assert 'SCORE_A2_R' in apply_in.columns, '27R을 먼저 실행하세요'

ent_age = pd.to_numeric(ent_t['AGE_BAND'], errors='coerce')
app_age = pd.to_numeric(apply_in['AGE_BAND'], errors='coerce')
oof_s   = pd.Series(oof_j)                     # 진입자 OOF (27R 결합 인코딩 기준)
app_s   = apply_in['SCORE_A2_R']

# ── ① 같은 연령대 안에서 진입자 vs 잔류자 ──
rows = []
for band in sorted(ent_age.dropna().unique()):
    e = oof_s[ (ent_age == band).values ]
    a = app_s[ (app_age == band).values ]
    if len(e) < 100 or len(a) < 100: continue
    rows.append({'연령대': int(band), '진입자 n': len(e), '잔류자 n': len(a),
                 '진입자 중앙값': round(float(e.median()), 5),
                 '잔류자 중앙값': round(float(a.median()), 5),
                 '잔류자/진입자 중앙값 배율': round(float(a.median() / max(e.median(), 1e-9)), 2),
                 '잔류자 중 진입자중앙값 초과 비율': f'{float((a > e.median()).mean())*100:.1f}%'})
res_age = pd.DataFrame(rows); display(res_age)
save_ckpt(res_age, 'step29_연령층화')

# ── ② 연령 표준화: 잔류자를 진입자 연령 조성으로 재가중 → 십분위 분포 재계산 ──
ent_comp = ent_age.value_counts(normalize=True)
app_comp = app_age.value_counts(normalize=True)
w = app_age.map(ent_comp / app_comp).fillna(0).values      # 사후층화 가중치
ent_q29 = np.quantile(oof_j, np.arange(0.1, 1.0, 0.1))
band10 = np.digitize(app_s.values, ent_q29)
wdist = pd.Series(w).groupby(pd.Series(band10)).sum(); wdist = wdist / wdist.sum() * 100
raw   = pd.Series(band10).value_counts(normalize=True).sort_index() * 100
tbl = pd.DataFrame({'원분포(%)': raw.round(2), '연령표준화 후(%)': wdist.round(2)})
tbl.index = [f'진입자 위험 {i+1}분위' for i in tbl.index]
display(tbl)
low3_raw = float(raw.reindex([0, 1, 2]).fillna(0).sum())
low3_std = float(wdist.reindex([0, 1, 2]).fillna(0).sum())
print(f'하위 3분위 비중: 원 {low3_raw:.1f}% → 연령 표준화 후 {low3_std:.1f}%  (균등 30%)')
print('판독: 표준화 후 30%에 근접하면 "쏠림은 대부분 연령 조성 효과" / 여전히 낮으면 "연령 외 신호 존재".')
print('①의 배율이 연령대 전반에서 1.0 근처면 같은 결론을 개별 연령대에서도 재확인하는 것.')


,연령대,진입자 n,잔류자 n,진입자 중앙값,잔류자 중앙값,잔류자/진입자 중앙값 배율,잔류자 중 진입자중앙값 초과 비율
0,2,51869,283944,0.00569,0.00460,0.81,45.4%
1,3,5904,81738,0.03319,0.02326,0.70,39.1%
2,4,4186,87920,0.05024,0.04173,0.83,45.1%
3,5,3226,112164,0.03544,0.02695,0.76,41.3%
4,6,1576,101222,0.02729,0.01812,0.66,42.2%
5,7,1324,82712,0.01695,0.01213,0.72,37.7%
6,8,525,40740,0.01700,0.01200,0.71,31.4%


[ckpt 저장] step29_연령층화: 7행 → /content/drive/MyDrive/09.개인_CB정보/ckpt/step29_연령층화.parquet


,원분포(%),연령표준화 후(%)
진입자 위험 1분위,5.43,11.46
진입자 위험 2분위,5.16,10.55
진입자 위험 3분위,6.39,12.82
진입자 위험 4분위,5.82,8.82
진입자 위험 5분위,6.58,8.48
진입자 위험 6분위,9.90,9.86
진입자 위험 7분위,16.03,14.40
진입자 위험 8분위,11.95,6.99
진입자 위험 9분위,18.43,8.92
진입자 위험 10분위,14.31,7.70


하위 3분위 비중: 원 17.0% → 연령 표준화 후 34.8%  (균등 30%)
판독: 표준화 후 30%에 근접하면 "쏠림은 대부분 연령 조성 효과" / 여전히 낮으면 "연령 외 신호 존재".
①의 배율이 연령대 전반에서 1.0 근처면 같은 결론을 개별 연령대에서도 재확인하는 것.


In [ ]:
# ============ STEP 30. 인구변수 제외 민감도 — "대안변수 순수 성능" + 성별 사용 이슈 방어 ============
# 기여도 45%가 인구(연령+성별)였다. 두 가지 질문에 답한다:
#  (a) 실무에서 사용이 제한되는 성별을 빼면 성능이 얼마나 빠지는가
#  (b) 인구를 전부 빼고 "순수 대안변수(생활이력·자산·평형)"만으로도 변별력이 성립하는가
#      → (b)의 CI 하한이 0.5를 넘으면 "대안변수 자체의 기여"가 인구와 독립적으로 입증된다.
GEN_COLS = [c for c in X_tr_j.columns if c.startswith('GENDER')]
AGE_COLS = [c for c in X_tr_j.columns if c.startswith('AGE')]
CONFIGS = {
    'F0 전체(공식 모델)':            X_tr_j,
    'F1 성별 제외':                  X_tr_j.drop(columns=GEN_COLS),
    'F2 인구 전부 제외(순수 대안변수)': X_tr_j.drop(columns=GEN_COLS + AGE_COLS),
}
rows = []
for name, Xv in CONFIGS.items():
    mu, sd, (alo, ahi), ap_s, _ = eval_seeds(Xv, y24, fp_final_factory(), HOLDOUT_SEEDS_25)
    rows.append({'구성': name, '피처 수': Xv.shape[1], 'AUROC(홀드아웃)': f'{mu:.4f}±{sd:.4f}',
                 'AUROC 95% CI': f'[{alo:.3f}, {ahi:.3f}]', 'AUPRC': round(ap_s, 4)})
res_demo = pd.DataFrame(rows); display(res_demo)
save_ckpt(res_demo, 'step30_인구제외민감도')

# F2(순수 대안변수)에서 무엇이 일하는지 — 상위 기여도
imps = []
fp_i = fp_final_factory(collect=imps)
_ = cv_oof_custom(CONFIGS['F2 인구 전부 제외(순수 대안변수)'], y24, groups, fp_i, seed=HOLDOUT_SEEDS_25[0])
imp = pd.concat(imps, axis=1).mean(axis=1); imp = (imp / imp.sum() * 100).sort_values(ascending=False)
display(imp.head(8).round(2).to_frame('기여도(gain %)'))
print('판독: F1이 F0과 CI가 겹치면 "성별 없이도 성능 유지" → 실무 전환 가능성 방어.')
print('      F2의 CI 하한 > 0.5면 "인구 효과를 걷어내도 생활이력·자산 정보에 변별력 존재" — 대안변수 기여의 최종 증거.')
print('      발표 권고: 공식 수치는 F0, 방법론 슬라이드에 F1·F2를 민감도로 병기.')


,구성,피처 수,AUROC(홀드아웃),AUROC 95% CI,AUPRC
0,F0 전체(공식 모델),21,0.8198±0.0057,"[0.788, 0.859]",0.0113
1,F1 성별 제외,19,0.7923±0.0051,"[0.760, 0.831]",0.0090
2,F2 인구 전부 제외(순수 대안변수),10,0.6790±0.0030,"[0.638, 0.721]",0.0067


[ckpt 저장] step30_인구제외민감도: 3행 → /content/drive/MyDrive/09.개인_CB정보/ckpt/step30_인구제외민감도.parquet


,기여도(gain %)
AL012G019_prev,21.34
U81301010_prev,19.63
AS120G001_prev,19.49
AL012G005_prev,11.95
U81306010_prev,10.00
AL012G011_prev,6.96
DELTA_AL012G005_prev,4.30
DELTA_AL012G019_prev,3.42


판독: F1이 F0과 CI가 겹치면 "성별 없이도 성능 유지" → 실무 전환 가능성 방어.
      F2의 CI 하한 > 0.5면 "인구 효과를 걷어내도 생활이력·자산 정보에 변별력 존재" — 대안변수 기여의 최종 증거.
      발표 권고: 공식 수치는 F0, 방법론 슬라이드에 F1·F2를 민감도로 병기.


## STEP 30R. F2 재튜닝 — 실무 정합 공식 수치 확정

**실행 경로**
- 런타임이 살아 있으면(30까지 돌린 그 세션): 이 셀 하나만 실행. 소요 5~15분.
- 새 런타임이면: **17 → 18 → 19-1~4 → 20 → 21 → 이 셀** 순서로 실행.

**시간 절약 팁 (새 런타임일 때)**: STEP 20과 21에서 시간이 걸리는 건 맨 아래 결과표 루프뿐이고,
30R에 필요한 정의(X2, 함수들)는 그 위에서 끝난다. 그래서
- STEP 20: `results = []`부터 셀 끝까지 주석 처리(#) 후 실행
- STEP 21: `rows = []`부터 셀 끝까지 주석 처리 후 실행
하면 준비가 2~3분으로 줄어든다. (주석 처리 안 하고 그냥 다 돌려도 되고, 그 경우 15~25분.)

필요 객체: `X2`, `ent_t`, `groups`, `LGB_SMALL`, `cv_oof_custom`, `cluster_bootstrap_ci`
(21R·21T·25A~30은 불필요 — 이 셀은 자체 함수로 독립 실행)

In [ ]:
# ============ STEP 30R. F2(인구 전부 제외) 구성 재튜닝 — 실무 정합 공식 수치 확정 ============
# 배경: 은행연합회 차별금지 모범규준(2012~)과 CB사 공시 관행에 따라 인구변수(성별·연령) 전면 배제
#       구성(F2)을 공식 수치로 승격. 기존 F2 0.679는 인구 컬럼만 뺀 측정(파라미터는 F0 기준)이므로,
#       F2 피처셋 기준으로 비율·파라미터를 다시 탐색해 정직한 최적치를 확정한다.
# 구조: 21T와 동일 3단 — ①탐색(seed 2개) ②선택(μ−σ) ③홀드아웃 seed로 공식 수치 측정.

from sklearn.metrics import roc_auc_score, average_precision_score

X2F = X2.drop(columns=[c for c in X2.columns if c.startswith(('GENDER', 'AGE'))])
y24r = ent_t['TARGET_24M'].astype(int).values
print(f'F2 피처셋: {X2F.shape[1]}개 (인구 더미 {X2.shape[1]-X2F.shape[1]}개 제거)')

TUNE_SEEDS_R, HOLDOUT_SEEDS_R = [42, 7], [101, 202]

def make_fp_down_r(ratio, params, n_models=5):
    def fp(Xtr, ytr, Xva):
        pos, neg = np.where(ytr == 1)[0], np.where(ytr == 0)[0]
        preds = []
        for s in range(n_models):
            rng = np.random.default_rng(s)
            sel = np.concatenate([pos, rng.choice(neg, size=min(len(neg), int(len(pos) * ratio)), replace=False)])
            m = lgb.LGBMClassifier(**{**params, 'random_state': 500 + s})
            m.fit(Xtr.iloc[sel], ytr[sel])
            preds.append(m.predict_proba(Xva)[:, 1])
        return np.mean(preds, axis=0)
    return fp

def eval_r(fp, seeds):
    aucs = []
    for sd in seeds:
        oof = cv_oof_custom(X2F, y24r, groups, fp, seed=sd)
        aucs.append(roc_auc_score(y24r, oof))
    return float(np.mean(aucs)), float(np.std(aucs))

LGB_D20 = {**LGB_SMALL, 'min_child_samples': 20}

print('=== ① 다운샘플 비율 (파라미터 고정: 현행 d2/leaf4/child20) ===')
P_BASE = {**LGB_D20, 'max_depth': 2, 'num_leaves': 4}
rows = []
for ratio in [10, 20, 50, 100]:
    mu, sd = eval_r(make_fp_down_r(ratio, P_BASE), TUNE_SEEDS_R)
    rows.append({'비율': f'{ratio}:1', 'AUROC 평균': round(mu, 4), '표준편차': round(sd, 4),
                 '안정성점수(μ−σ)': round(mu - sd, 4)})
res_r1 = pd.DataFrame(rows); display(res_r1)
BEST_RATIO_R = int(res_r1.iloc[res_r1['안정성점수(μ−σ)'].idxmax()]['비율'].split(':')[0])
print(f'→ 선택: {BEST_RATIO_R}:1')

print('\n=== ② 트리 파라미터 (비율 고정) ===')
GRID_R = {
    'R1 d2/leaf4/child20 (F0 최적)':  P_BASE,
    'R2 d3/leaf7/child20':            LGB_D20,
    'R3 d2/leaf4 + 낮은lr·많은트리':   {**P_BASE, 'learning_rate': 0.02, 'n_estimators': 800},
    'R4 d2/leaf4 + 강한 L2(reg=20)':  {**P_BASE, 'reg_lambda': 20.0},
    'R5 d3/leaf7 + child50':          {**LGB_D20, 'min_child_samples': 50},
}
rows = []
for name, P in GRID_R.items():
    mu, sd = eval_r(make_fp_down_r(BEST_RATIO_R, P), TUNE_SEEDS_R)
    rows.append({'설정': name, 'AUROC 평균': round(mu, 4), '표준편차': round(sd, 4),
                 '안정성점수(μ−σ)': round(mu - sd, 4)})
res_r2 = pd.DataFrame(rows); display(res_r2)
BEST_NAME_R = res_r2.iloc[res_r2['안정성점수(μ−σ)'].idxmax()]['설정']
print(f'→ 선택: {BEST_NAME_R}')

print('\n=== ③ 홀드아웃 seed 공식 측정 ===')
fp_fin = make_fp_down_r(BEST_RATIO_R, GRID_R[BEST_NAME_R])
mu_t, sd_t = eval_r(fp_fin, TUNE_SEEDS_R)
mu_h, sd_h = eval_r(fp_fin, HOLDOUT_SEEDS_R)
oof_h = cv_oof_custom(X2F, y24r, groups, fp_fin, seed=HOLDOUT_SEEDS_R[0])
(am, alo, ahi), (pm, plo, phi) = cluster_bootstrap_ci(y24r, oof_h, groups)
res_r3 = pd.DataFrame([{
    '최종 F2 구성': f'다운샘플 {BEST_RATIO_R}:1 × 5앙상블 · {BEST_NAME_R}',
    '탐색 AUROC': f'{mu_t:.4f}±{sd_t:.4f}', '홀드아웃 AUROC': f'{mu_h:.4f}±{sd_h:.4f}',
    '낙관편의': round(mu_t - mu_h, 4), 'AUROC 95% CI': f'[{alo:.3f}, {ahi:.3f}]',
    'AUPRC': round(float(average_precision_score(y24r, oof_h)), 4),
    '재튜닝 전 F2': 0.679}])
display(res_r3)
save_ckpt(res_r1, 'step30R_비율탐색'); save_ckpt(res_r2, 'step30R_파라미터탐색'); save_ckpt(res_r3, 'step30R_최종확정')
print('\n[판독] 홀드아웃 값이 새 공식 수치. 0.679 대비 상승분 = 재튜닝 효과.')
print('       낙관편의 > 0.02면 탐색이 분할 운을 주운 것 — 홀드아웃 값만 보고.')
print('       확정 시 갱신할 곳: 문서 9절 F2 행, 엑셀 최종성능_v2 시트, 결론 문안의 0.679.')


F2 피처셋: 10개 (인구 더미 11개 제거)
=== ① 다운샘플 비율 (파라미터 고정: 현행 d2/leaf4/child20) ===


,비율,AUROC 평균,표준편차,안정성점수(μ−σ)
0,10:1,0.6883,0.0046,0.6837
1,20:1,0.6892,0.0061,0.6830
2,50:1,0.6925,0.0054,0.6871
3,100:1,0.6933,0.0063,0.6870


→ 선택: 50:1

=== ② 트리 파라미터 (비율 고정) ===


,설정,AUROC 평균,표준편차,안정성점수(μ−σ)
0,R1 d2/leaf4/child20 (F0 최적),0.6925,0.0054,0.6871
1,R2 d3/leaf7/child20,0.6946,0.0024,0.6922
2,R3 d2/leaf4 + 낮은lr·많은트리,0.6927,0.0060,0.6867
3,R4 d2/leaf4 + 강한 L2(reg=20),0.6862,0.0055,0.6806
4,R5 d3/leaf7 + child50,0.6934,0.0036,0.6897


→ 선택: R2 d3/leaf7/child20

=== ③ 홀드아웃 seed 공식 측정 ===


,최종 F2 구성,탐색 AUROC,홀드아웃 AUROC,낙관편의,AUROC 95% CI,AUPRC,재튜닝 전 F2
0,다운샘플 50:1 × 5앙상블 · R2 d3/leaf7/child20,0.6946±0.0024,0.6748±0.0051,0.0198,"[0.638, 0.720]",0.0061,0.679


[ckpt 저장] step30R_비율탐색: 4행 → /content/drive/MyDrive/09.개인_CB정보/ckpt/step30R_비율탐색.parquet
[ckpt 저장] step30R_파라미터탐색: 5행 → /content/drive/MyDrive/09.개인_CB정보/ckpt/step30R_파라미터탐색.parquet
[ckpt 저장] step30R_최종확정: 1행 → /content/drive/MyDrive/09.개인_CB정보/ckpt/step30R_최종확정.parquet

[판독] 홀드아웃 값이 새 공식 수치. 0.679 대비 상승분 = 재튜닝 효과.
       낙관편의 > 0.02면 탐색이 분할 운을 주운 것 — 홀드아웃 값만 보고.
       확정 시 갱신할 곳: 문서 9절 F2 행, 엑셀 최종성능_v2 시트, 결론 문안의 0.679.


## STEP 28. 산출물 v2 일괄 갱신

In [ ]:
# ============ STEP 28. 산출물 v2 일괄 갱신 — 학습파일 2종(원값) + 데이터 사전 ============
# 모델링 담당 전달 세트를 최종 피처 기준으로 재생성한다.

# ── 학습파일 (원값 버전 — 트리 모델용) ──
A2_RAW = [c for c in ent_t.columns if (c.endswith('_prev') and not c.startswith('PERF')
          and c.replace('_prev', '') != 'AL0C00001')]
A2_RAW = [c for c in A2_RAW if c.replace('_prev','') in
          set(BASE_V2) | set(AL3) | {'COMBO_JEONSE','COMBO_ASSET','CAR_FLAG'} |
          {f'DELTA_{a}' for a in AL3} | {p + '_CONF_MISSING' for p in
           ['U81301010','U81302010','U81201010']}]
A2_RAW = sorted(set(A2_RAW) | {c for c in X2_FINAL.columns if c in ent_t.columns})
A1_RAW = sorted(set(BASE_V2 + FLAGS + [f'DELTA_{c}' for c in AL3] + ['NONBANK_RATIO']))

meta = ['ID', 'YEAR', 'GENDER', 'AGE_BAND', 'TARGET_12M', 'TARGET_24M', 'HAS_NEXT']
for fname, cols in [('track_a_train_A1_v2.csv', A1_RAW), ('track_a_train_A2_v2.csv', A2_RAW)]:
    use = meta + [c for c in cols if c in ent_t.columns and c not in meta]
    ent_t[use].rename(columns={'YEAR': 'ENTRY_YEAR'}).to_csv(os.path.join(DATA_DIR, fname), index=False)
    print(f'저장: {fname} ({len(ent_t):,}행 × {len(use)}열)')

# ── 데이터 사전 v2 ──
KOR = {'AL012G005': '주소변동이력(3년누적, 구간)', 'AL012G011': '직장변동이력(3년누적, 구간)',
       'AL012G019': '휴대폰변동이력(3년누적, 구간)', 'U81301010': '거주지 매매가',
       'U81305010': '거주지 매매가(구간)', 'U81306010': '거주지 전세가(구간)', 'U81302010': '거주지 전세가',
       'U81201010': '총자산 평가금액', 'U81202010': '총자산 평가금액(구간)', 'U81102010': '보유주택 매매가',
       'AS120G001': '현거주지 평형(정보없음=독립범주)', 'GENDER': '성별', 'AGE_BAND': '연령대',
       'NONBANK_RATIO': '[파생] 2금융권 대출 의존비율(A-1 전용)',
       'CAR_FLAG': '[파생] 차량정보 보유 플래그', 'CAR_FLAG_prev': '[파생] 차량정보 보유 플래그(t-1)',
       'COMBO_JEONSE_prev': '[파생] 전세가순위×신뢰서열(t-1)', 'COMBO_ASSET_prev': '[파생] 자산순위×신뢰서열(t-1)',
       'TARGET_12M': '[타겟] 진입 후 12개월 내 30일+ 연체', 'TARGET_24M': '[타겟] 진입 후 24개월 내 30일+ 연체(2022 진입자는 12M 검열)',
       'HAS_NEXT': 't+1 스냅샷 관측 여부(검열 플래그)'}
def kor_name(c):
    base = c.replace('_prev', '').replace('DELTA_', '')
    tag  = ' — Δ최근1년' if c.startswith('DELTA_') else ''
    tag += ' — t-1(씬파일러 시절)' if c.endswith('_prev') else ''
    if c.endswith('_CONF_MISSING') or c.endswith('_CONF_MISSING_prev'):
        return f'[파생] {KOR.get(base.replace("_CONF_MISSING",""), base)} 신뢰정보없음 플래그{tag}'
    return KOR.get(c, KOR.get(base, base) + tag)

all_cols = sorted(set(A1_RAW) | set(A2_RAW) | set(meta))
dic = pd.DataFrame({'컬럼': all_cols,
                    '한글명/설명': [kor_name(c) for c in all_cols],
                    '사용처': [('A-1' if c in A1_RAW else '') + ('/A-2' if c in A2_RAW else '') or '공통(메타)'
                              for c in all_cols]})
dic.to_csv(os.path.join(DATA_DIR, '데이터사전_v2.csv'), index=False, encoding='utf-8-sig')
print(f'저장: 데이터사전_v2.csv ({len(dic)}행)')

print("""
── 남은 문서 수정 체크리스트 (코드 아님, 수동) ──
1. 타겟변수 문서 7절: TARGET_24M 정의 + STEP 16(가설 B) 연결 문단 추가
2. 대안변수 문서 3·7절: 신뢰수준 3종 재배치(플래그), CAR_FLAG·Δ주소 채택, U10000002 기각 기록
3. 대안변수 문서 9절: STEP 25A 공식 표(res25)로 최종 성능 기입
4. 인수인계 요약 5절: 불균형 방침 갱신 — 'spw' → '다운샘플 50:1×5앙상블(21R/21T 실측 근거), SMOTE 배제 유지'
5. 한글명대조표 엑셀: 데이터사전_v2.csv 내용 반영
""")


저장: track_a_train_A1_v2.csv (68,677행 × 22열)
저장: track_a_train_A2_v2.csv (68,677행 × 23열)
저장: 데이터사전_v2.csv (38행)

── 남은 문서 수정 체크리스트 (코드 아님, 수동) ──
1. 타겟변수 문서 7절: TARGET_24M 정의 + STEP 16(가설 B) 연결 문단 추가
2. 대안변수 문서 3·7절: 신뢰수준 3종 재배치(플래그), CAR_FLAG·Δ주소 채택, U10000002 기각 기록
3. 대안변수 문서 9절: STEP 25A 공식 표(res25)로 최종 성능 기입
4. 인수인계 요약 5절: 불균형 방침 갱신 — 'spw' → '다운샘플 50:1×5앙상블(21R/21T 실측 근거), SMOTE 배제 유지'
5. 한글명대조표 엑셀: 데이터사전_v2.csv 내용 반영



In [ ]:
print(ASSET_KEEP)

['U81301010', 'U81305010', 'U81306010']


In [ ]:
import pandas as pd, os, glob

DIR = '/content/drive/MyDrive/Track A 전처리 파일/Track A 수정 파일'

# 1) 실제 파일명 확인 (화면에서 이름이 잘려 보이니 먼저 확인)
for f in sorted(glob.glob(os.path.join(DIR, '*.csv'))):
    print(os.path.basename(f))

# 2) A-2 원값 파일 컬럼 확인
path = os.path.join(DIR, 'track_a_train_A2_v2.csv')
cols = pd.read_csv(path, nrows=1).columns.tolist()
print(f'\n총 {len(cols)}개 컬럼\n{cols}')

track_a_apply_v2_scored_R.csv
track_a_train_A1_binned_v2_final.csv
track_a_train_A1_v2.csv
track_a_train_A2_binned_v2_final.csv
track_a_train_A2_v2.csv
데이터사전_v2.csv

총 23개 컬럼
['ID', 'ENTRY_YEAR', 'GENDER', 'AGE_BAND', 'TARGET_12M', 'TARGET_24M', 'HAS_NEXT', 'AL012G005_prev', 'AL012G011_prev', 'AL012G019_prev', 'AS120G001_prev', 'CAR_FLAG_prev', 'COMBO_ASSET_prev', 'COMBO_JEONSE_prev', 'DELTA_AL012G005_prev', 'DELTA_AL012G011_prev', 'DELTA_AL012G019_prev', 'U81201010_CONF_MISSING_prev', 'U81301010_CONF_MISSING_prev', 'U81301010_prev', 'U81302010_CONF_MISSING_prev', 'U81305010_prev', 'U81306010_prev']


In [ ]:
import pandas as pd, os, glob

DIR = '/content/drive/MyDrive/Track A 전처리 파일/Track A 수정 파일'

# 1) 실제 파일명 확인 (화면에서 이름이 잘려 보이니 먼저 확인)
for f in sorted(glob.glob(os.path.join(DIR, '*.csv'))):
    print(os.path.basename(f))

# 2) A-1 원값 파일 컬럼 확인
path = os.path.join(DIR, 'track_a_train_A1_v2.csv')
cols = pd.read_csv(path, nrows=1).columns.tolist()
print(f'\n총 {len(cols)}개 컬럼\n{cols}')

track_a_apply_v2_scored_R.csv
track_a_train_A1_binned_v2_final.csv
track_a_train_A1_v2.csv
track_a_train_A2_binned_v2_final.csv
track_a_train_A2_v2.csv
데이터사전_v2.csv

총 22개 컬럼
['ID', 'ENTRY_YEAR', 'GENDER', 'AGE_BAND', 'TARGET_12M', 'TARGET_24M', 'HAS_NEXT', 'AL012G005', 'AL012G011', 'AL012G019', 'AS120G001', 'CAR_FLAG', 'DELTA_AL012G005', 'DELTA_AL012G011', 'DELTA_AL012G019', 'NONBANK_RATIO', 'U81201010_CONF_MISSING', 'U81301010', 'U81301010_CONF_MISSING', 'U81302010_CONF_MISSING', 'U81305010', 'U81306010']


In [ ]:
import pandas as pd, os

DIR = '/content/drive/MyDrive/Track A 전처리 파일/Track A 수정 파일'
META = {'ID','ENTRY_YEAR','GENDER','AGE_BAND','TARGET_12M','TARGET_24M','HAS_NEXT'}

a1 = [c for c in pd.read_csv(os.path.join(DIR,'track_a_train_A1_v2.csv'), nrows=1).columns if c not in META]
a2 = [c for c in pd.read_csv(os.path.join(DIR,'track_a_train_A2_v2.csv'), nrows=1).columns if c not in META]

print(f"A-1 피처 {len(a1)}개 중 _prev 붙은 것: {sum(c.endswith('_prev') for c in a1)}개  (0이어야 정상)")
print(f"A-2 피처 {len(a2)}개 중 _prev 붙은 것: {sum(c.endswith('_prev') for c in a2)}개  (전부여야 정상)")

# 같은 변수가 두 파일에서 어떤 이름인지 대조
base2 = {c[:-5] for c in a2}                      # _prev 떼어낸 기준명
print("\n기준명          | A-1 파일        | A-2 파일")
for b in sorted(base2 & set(a1)):
    print(f"{b:<15} | {b:<15} | {b}_prev")
print(f"\nA-1 전용(=A-2에 없음): {sorted(set(a1) - base2)}")

A-1 피처 15개 중 _prev 붙은 것: 0개  (0이어야 정상)
A-2 피처 16개 중 _prev 붙은 것: 16개  (전부여야 정상)

기준명          | A-1 파일        | A-2 파일
AL012G005       | AL012G005       | AL012G005_prev
AL012G011       | AL012G011       | AL012G011_prev
AL012G019       | AL012G019       | AL012G019_prev
AS120G001       | AS120G001       | AS120G001_prev
CAR_FLAG        | CAR_FLAG        | CAR_FLAG_prev
DELTA_AL012G005 | DELTA_AL012G005 | DELTA_AL012G005_prev
DELTA_AL012G011 | DELTA_AL012G011 | DELTA_AL012G011_prev
DELTA_AL012G019 | DELTA_AL012G019 | DELTA_AL012G019_prev
U81201010_CONF_MISSING | U81201010_CONF_MISSING | U81201010_CONF_MISSING_prev
U81301010       | U81301010       | U81301010_prev
U81301010_CONF_MISSING | U81301010_CONF_MISSING | U81301010_CONF_MISSING_prev
U81302010_CONF_MISSING | U81302010_CONF_MISSING | U81302010_CONF_MISSING_prev
U81305010       | U81305010       | U81305010_prev
U81306010       | U81306010       | U81306010_prev

A-1 전용(=A-2에 없음): ['NONBANK_RATIO']


## [최종 검증] 미확인 산출물 점검

데이터사전·구간화 파일·apply 점수 파일은 자동 생성 후 내용을 확인한 적이 없다. 전달 전에 한 번 돌려 이상 유무만 보는 셀.

**독립 실행 가능** — 다른 STEP 없이 이 셀만 실행하면 된다(파일만 읽음). 몇 초 소요.

In [ ]:
# ============ [최종 검증] 미확인 산출물 2종 점검 — 데이터사전 / apply 점수 파일 ============
# 이 두 파일은 자동 생성 후 아무도 열어보지 않았다. 전달 전 마지막 확인.
import pandas as pd, os
DIR = '/content/drive/MyDrive/Track A 전처리 파일/Track A 수정 파일'

# ── ① 데이터사전: 자동 생성된 한글명이 깨지지 않았는지 ──
dic = pd.read_csv(os.path.join(DIR, '데이터사전_v2.csv'))
print(f'[데이터사전] {len(dic)}행 / 컬럼: {list(dic.columns)}')
bad = dic[dic.iloc[:, 1].astype(str).str.strip().eq(dic.iloc[:, 0].astype(str).str.strip())]
print(f'한글명이 영문컬럼명 그대로인 행(설명 실패): {len(bad)}개')
if len(bad): display(bad)
display(dic.head(12))

# ── ② 구간화 파일: 재실행본 컬럼 구성 ──
for f in ['track_a_train_A1_binned_v2_final.csv', 'track_a_train_A2_binned_v2_final.csv']:
    c = pd.read_csv(os.path.join(DIR, f), nrows=1).columns.tolist()
    META = {'ID','ENTRY_YEAR','GENDER','AGE_BAND','TARGET_12M','TARGET_24M','HAS_NEXT'}
    print(f'\n[{f}] {len(c)}열 | 피처 {len([x for x in c if x not in META])}개')
    print('  피처:', [x for x in c if x not in META])

# ── ③ apply 점수 파일: 행수·결측·분포 ──
ap = pd.read_csv(os.path.join(DIR, 'track_a_apply_v2_scored_R.csv'))
print(f'\n[apply] {len(ap):,}행 (기대 794,773) | 결측 {ap.isna().sum().sum()}개')
print(f'  엄격 씬파일러 {int(ap.IS_STRICT.sum()):,}명 (기대 333,571)')
print(f'  ID 중복: {ap.ID.duplicated().sum()}개 (0이어야 정상)')
display(ap['SCORE_A2'].describe().to_frame('SCORE_A2 분포'))
print('  점수 범위가 0~1 안에 있고 중복 ID가 없으면 정상. 절대값은 미보정이므로 순위 용도로만.')


[데이터사전] 38행 / 컬럼: ['컬럼', '한글명/설명', '사용처']
한글명이 영문컬럼명 그대로인 행(설명 실패): 2개


,컬럼,한글명/설명,사용처
21,ID,ID,공통(메타)
37,YEAR,YEAR,공통(메타)


,컬럼,한글명/설명,사용처
0,AGE_BAND,연령대,공통(메타)
1,AL012G005,"주소변동이력(3년누적, 구간)",A-1
2,AL012G005_prev,"주소변동이력(3년누적, 구간) — t-1(씬파일러 시절)",/A-2
3,AL012G011,"직장변동이력(3년누적, 구간)",A-1
4,AL012G011_prev,"직장변동이력(3년누적, 구간) — t-1(씬파일러 시절)",/A-2
5,AL012G019,"휴대폰변동이력(3년누적, 구간)",A-1
6,AL012G019_prev,"휴대폰변동이력(3년누적, 구간) — t-1(씬파일러 시절)",/A-2
7,AS120G001,현거주지 평형(정보없음=독립범주),A-1
8,AS120G001_prev,현거주지 평형(정보없음=독립범주) — t-1(씬파일러 시절),/A-2
9,CAR_FLAG,[파생] 차량정보 보유 플래그,A-1



[track_a_train_A1_binned_v2_final.csv] 21열 | 피처 15개
  피처: ['AL012G019', 'U81301010', 'U81305010', 'AS120G001', 'U81306010', 'AL012G005', 'AL012G011', 'U81301010_CONF_MISSING', 'U81302010_CONF_MISSING', 'U81201010_CONF_MISSING', 'CAR_FLAG', 'DELTA_AL012G005', 'DELTA_AL012G011', 'DELTA_AL012G019', 'NONBANK_RATIO']

[track_a_train_A2_binned_v2_final.csv] 16열 | 피처 10개
  피처: ['AL012G019_prev', 'U81301010_prev', 'U81305010_prev', 'AS120G001_prev', 'U81306010_prev', 'AL012G005_prev', 'AL012G011_prev', 'DELTA_AL012G005_prev', 'DELTA_AL012G011_prev', 'DELTA_AL012G019_prev']

[apply] 794,773행 (기대 794,773) | 결측 0개
  엄격 씬파일러 333,571명 (기대 333,571)
  ID 중복: 0개 (0이어야 정상)


,SCORE_A2 분포
count,794773.000000
mean,0.025416
std,0.032526
min,0.000552
25%,0.006722
50%,0.012135
75%,0.034299
max,0.537741


  점수 범위가 0~1 안에 있고 중복 ID가 없으면 정상. 절대값은 미보정이므로 순위 용도로만.
